# JaCoCo + Static DU — White Box Validation Notebook

End-to-end execution of **JaCoCo** and **Static DU** against a Java repository. Preserves every raw tool output exactly as emitted and validates **31+ White Box metrics** across:

- Control Flow Testing
- Test Regression / Coverage Analysis
- Data Flow Testing

**Default repository:** [java-tool-testing-def-use](https://github.com/visvantha-testable/java-tool-testing-def-use)

Each metric is classified as **Directly Emitted**, **Derived**, or **Not Supported** — never fabricated.


## Section 1 — Install Dependencies


In [ ]:
import platform
import shutil
import subprocess
import sys

IS_COLAB = 'google.colab' in sys.modules
IS_LINUX = platform.system() == 'Linux'

if IS_COLAB or IS_LINUX:
    !apt-get update -qq
    !apt-get install -y openjdk-17-jdk maven git

!pip install -q pandas gitpython lxml notebook jupyter

print('Java:')
subprocess.run(['java', '-version'], check=False)
if shutil.which('mvn'):
    print('\nMaven:')
    subprocess.run(['mvn', '-version'], check=False)


## Section 2 — Configuration


In [ ]:
# Mode 1: Git clone
USE_GIT_REPO = True
USE_GIT = USE_GIT_REPO  # alias

REPO_URL = 'https://github.com/visvantha-testable/java-tool-testing-def-use.git'

# Mode 2: Local path
LOCAL_REPO = './workspace/java-tool-testing-def-use'

WORKSPACE_DIR = './workspace'
OUTPUT_DIR = './outputs'
IF_CLONE_EXISTS = 'reuse'
CLONE_DEPTH = 1
SKIP_VERIFY = True
BASELINE_JACOCO_XML = ''

# Colab example:
# USE_GIT_REPO = False
# LOCAL_REPO = '/content/java-tool-testing-def-use'


## Section 3 — Imports and Utility Functions


In [ ]:
import json
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import display

JACOCO_UTILS_ROOT = Path('..') / 'JaCoCo Coverage' / 'tool'
JACOCO_PATH_UTILS_ROOT = Path('..') / 'JaCoCo Path Analysis' / 'tool'
STATIC_DU_UTILS_ROOT = Path('..') / 'Static DU Analysis' / 'tool'
for path in (JACOCO_UTILS_ROOT, JACOCO_PATH_UTILS_ROOT, STATIC_DU_UTILS_ROOT, Path('tool')):
    sys.path.insert(0, str(path.resolve()))

from __future__ import annotations

import csv
import os
import platform
import re
import shutil
import subprocess
import sys
import time
import xml.etree.ElementTree as ET
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from git import Repo
from git.exc import GitCommandError, InvalidGitRepositoryError

EXCLUDED_DIR_NAMES = {
    ".git", "target", "build", "bin", ".idea", ".gradle", ".mvn", "out", "node_modules",
}
COUNTER_TYPES = ["INSTRUCTION", "BRANCH", "LINE", "METHOD", "CLASS", "COMPLEXITY"]
PY = sys.executable
PACKAGE_RE = re.compile(r"^\s*package\s+([\w.]+)\s*;", re.MULTILINE)


@dataclass
class CommandResult:
    command: list[str]
    stdout: str
    stderr: str
    exit_code: int
    execution_time_seconds: float


@dataclass
class BuildStatus:
    build_tool: str
    build_command: list[str]
    jacoco_command: list[str]
    build_result: CommandResult | None = None
    jacoco_result: CommandResult | None = None
    build_success: bool = False
    test_success: bool = False
    report_generated: bool = False
    report_dir: Path | None = None
    jacoco_exec: Path | None = None
    jacoco_xml: Path | None = None
    jacoco_csv: Path | None = None
    index_html: Path | None = None


class NotebookLogger:
    def __init__(self, error_log_path: Path) -> None:
        self.error_log_path = error_log_path
        self.error_log_path.parent.mkdir(parents=True, exist_ok=True)
        self._errors: list[dict[str, str]] = []
        self.write_errors()

    def info(self, message: str) -> None:
        timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        print(f"[{timestamp}] INFO: {message}")

    def error(self, message: str, step: str = "notebook") -> None:
        timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        line = f"[{timestamp}] ERROR: {message}\n"
        print(line, end="")
        self._errors.append({"timestamp": timestamp, "step": step, "error": message})
        self.write_errors()

    def write_errors(self) -> None:
        with self.error_log_path.open("w", encoding="utf-8", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=["timestamp", "step", "error"])
            writer.writeheader()
            writer.writerows(self._errors)


def ensure_output_dir(output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)


def project_runtimes_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    candidates: list[Path] = []
    for parent in [current, *current.parents]:
        candidate = parent / "runtimes"
        if candidate.is_dir():
            candidates.append(candidate)
    for candidate in candidates:
        if (candidate / "jdk-21").exists() or any(candidate.glob("apache-maven-*")):
            return candidate
    return candidates[0] if candidates else current / "runtimes"


def configure_java_runtime(logger: NotebookLogger) -> dict[str, str]:
    runtimes = project_runtimes_root()
    env = os.environ.copy()
    jdk_candidates = [
        runtimes / "jdk-21",
        Path(env.get("JAVA_HOME", "")),
        Path(r"C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot"),
    ]
    for candidate in jdk_candidates:
        java_bin = candidate / "bin"
        java_exe = java_bin / ("java.exe" if platform.system() == "Windows" else "java")
        if java_exe.exists():
            env["JAVA_HOME"] = str(candidate)
            env["PATH"] = str(java_bin) + os.pathsep + env.get("PATH", "")
            logger.info(f"Using JAVA_HOME={candidate}")
            break
    else:
        logger.info("Using system Java from PATH.")
    return env


def java_version_text(env: dict[str, str]) -> str:
    completed = subprocess.run(
        ["java", "-version"],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        check=False,
        env=env,
    )
    return combine_streams(completed.stdout, completed.stderr).strip()


def resolve_maven_command(repo_path: Path, logger: NotebookLogger) -> list[str]:
    if platform.system() == "Windows":
        wrapper = repo_path / "mvnw.cmd"
        if wrapper.exists():
            return [str(wrapper)]
    else:
        wrapper = repo_path / "mvnw"
        if wrapper.exists():
            return [str(wrapper)]
    runtimes = project_runtimes_root(repo_path)
    bundled = runtimes / "apache-maven-3.9.6" / "bin" / ("mvn.cmd" if platform.system() == "Windows" else "mvn")
    if bundled.exists():
        logger.info(f"Using bundled Maven: {bundled}")
        return [str(bundled)]
    return ["mvn"]


def resolve_gradle_command(repo_path: Path, logger: NotebookLogger) -> list[str]:
    if platform.system() == "Windows":
        wrapper = repo_path / "gradlew.bat"
    else:
        wrapper = repo_path / "gradlew"
    if wrapper.exists():
        if platform.system() != "Windows":
            wrapper.chmod(wrapper.stat().st_mode | 0o111)
        logger.info(f"Using Gradle wrapper: {wrapper}")
        return [str(wrapper)]
    return ["gradle"]


def derive_clone_path(repo_url: str, workspace_dir: Path) -> Path:
    repo_name = repo_url.rstrip("/").removesuffix(".git").split("/")[-1]
    if not repo_name:
        raise ValueError(f"Unable to derive repository name from URL: {repo_url}")
    return workspace_dir / repo_name


def validate_repo_url(repo_url: str) -> None:
    if not repo_url or not isinstance(repo_url, str):
        raise ValueError("REPO_URL must be a non-empty string.")
    if not (repo_url.startswith("https://") or repo_url.startswith("git@") or repo_url.startswith("http://")):
        raise ValueError(f"Invalid repository URL format: {repo_url}")


def clone_or_reuse_repository(
    repo_url: str,
    workspace_dir: Path,
    if_clone_exists: str,
    logger: NotebookLogger,
    clone_depth: int | None = None,
) -> Path:
    validate_repo_url(repo_url)
    workspace_dir.mkdir(parents=True, exist_ok=True)
    clone_path = derive_clone_path(repo_url, workspace_dir)
    if clone_path.exists():
        if if_clone_exists == "reclone":
            logger.info(f"Removing existing clone at {clone_path}")
            shutil.rmtree(clone_path)
        elif if_clone_exists == "reuse":
            logger.info(f"Reusing existing clone at {clone_path}")
            return clone_path.resolve()
        else:
            raise ValueError('IF_CLONE_EXISTS must be either "reuse" or "reclone"')
    logger.info(f"Cloning {repo_url} into {clone_path}")
    clone_kwargs: dict[str, Any] = {"depth": clone_depth} if clone_depth else {}
    try:
        Repo.clone_from(repo_url, clone_path, **clone_kwargs)
    except GitCommandError as exc:
        logger.error(f"Git clone failed: {exc}", step="clone")
        raise
    return clone_path.resolve()


def validate_local_repo_path(local_repo_path: Path, logger: NotebookLogger) -> Path:
    if not local_repo_path.exists():
        msg = f"Local repository path does not exist: {local_repo_path}"
        logger.error(msg, step="repository")
        raise FileNotFoundError(msg)
    if not local_repo_path.is_dir():
        msg = f"Local repository path is not a directory: {local_repo_path}"
        logger.error(msg, step="repository")
        raise NotADirectoryError(msg)
    build_files = [
        local_repo_path / "pom.xml",
        local_repo_path / "build.gradle",
        local_repo_path / "build.gradle.kts",
    ]
    if not any(path.exists() for path in build_files):
        msg = "No pom.xml, build.gradle, or build.gradle.kts found in repository."
        logger.error(msg, step="repository")
        raise FileNotFoundError(msg)
    return local_repo_path.resolve()


def resolve_repository_path(
    use_git_repo: bool,
    repo_url: str,
    local_repo: str | Path,
    workspace_dir: Path,
    if_clone_exists: str,
    logger: NotebookLogger,
    clone_depth: int | None = None,
) -> Path:
    if use_git_repo:
        return clone_or_reuse_repository(repo_url, workspace_dir, if_clone_exists, logger, clone_depth)
    return validate_local_repo_path(Path(local_repo), logger)


def detect_build_tool(repo_path: Path) -> str:
    if (repo_path / "pom.xml").exists():
        return "Maven"
    if (repo_path / "build.gradle.kts").exists():
        return "Gradle Kotlin DSL"
    if (repo_path / "build.gradle").exists():
        return "Gradle"
    raise FileNotFoundError("Unable to detect Maven or Gradle build files.")


def discover_java_files(repo_path: Path) -> list[Path]:
    files: list[Path] = []
    for path in repo_path.rglob("*.java"):
        if any(part in EXCLUDED_DIR_NAMES for part in path.parts):
            continue
        files.append(path.resolve())
    return sorted(files)


def extract_package_name(java_path: Path) -> str:
    try:
        text = java_path.read_text(encoding="utf-8")
    except (OSError, UnicodeDecodeError):
        return ""
    match = PACKAGE_RE.search(text)
    return match.group(1) if match else ""


def save_java_inventory(java_files: list[Path], output_csv: Path) -> None:
    rows = [
        {
            "file_path": str(path),
            "file_name": path.name,
            "package": extract_package_name(path),
            "directory": str(path.parent),
        }
        for path in java_files
    ]
    pd.DataFrame(rows).to_csv(output_csv, index=False)


def count_all_files(repo_path: Path) -> int:
    total = 0
    for path in repo_path.rglob("*"):
        if path.is_file() and not any(part in EXCLUDED_DIR_NAMES for part in path.parts):
            total += 1
    return total


def compute_repository_stats(
    repo_path: Path,
    java_files: list[Path],
    build_tool: str,
    java_version: str,
) -> dict[str, Any]:
    total_size = sum(path.stat().st_size for path in java_files)
    return {
        "repository_name": repo_path.name,
        "repository_location": str(repo_path),
        "build_tool": build_tool,
        "java_version": java_version.replace("\n", " | "),
        "total_java_files": len(java_files),
        "total_files": count_all_files(repo_path),
        "repository_size_bytes": total_size,
    }


def combine_streams(stdout: str, stderr: str) -> str:
    raw = stdout
    if stderr:
        if raw and not raw.endswith("\n"):
            raw += "\n"
        raw += stderr
    return raw


def run_shell_command(
    command: list[str],
    cwd: Path,
    env: dict[str, str],
    logger: NotebookLogger,
    step: str,
) -> CommandResult:
    logger.info(f"Executing ({step}): {' '.join(command)}")
    started = time.perf_counter()
    completed = subprocess.run(
        command,
        cwd=str(cwd),
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        check=False,
        env=env,
    )
    elapsed = round(time.perf_counter() - started, 5)
    result = CommandResult(
        command=command,
        stdout=completed.stdout,
        stderr=completed.stderr,
        exit_code=completed.returncode,
        execution_time_seconds=elapsed,
    )
    if completed.returncode != 0:
        logger.error(
            combine_streams(completed.stdout, completed.stderr) or f"Command failed with exit code {completed.returncode}",
            step=step,
        )
    return result


def find_jacoco_report_dirs(repo_path: Path) -> list[Path]:
    report_dirs: list[Path] = []
    for xml_path in repo_path.rglob("jacoco.xml"):
        if ".git" in xml_path.parts:
            continue
        parent = str(xml_path.parent).replace("\\", "/")
        if "/site/jacoco" in parent or "/reports/jacoco" in parent or parent.endswith("/jacoco"):
            report_dirs.append(xml_path.parent.resolve())
    return sorted(set(report_dirs))


def find_jacoco_exec_files(repo_path: Path) -> list[Path]:
    exec_files: list[Path] = []
    for exec_path in repo_path.rglob("jacoco.exec"):
        if ".git" in exec_path.parts:
            continue
        exec_files.append(exec_path.resolve())
    return sorted(exec_files)


def choose_primary_report_dir(report_dirs: list[Path]) -> Path | None:
    if not report_dirs:
        return None
    best_dir = report_dirs[0]
    best_score = -1
    for report_dir in report_dirs:
        xml_path = report_dir / "jacoco.xml"
        if not xml_path.exists():
            continue
        counters = parse_counter_map(xml_path)
        score = sum(counters.get(counter, {}).get("covered", 0) + counters.get(counter, {}).get("missed", 0) for counter in COUNTER_TYPES)
        if score > best_score:
            best_score = score
            best_dir = report_dir
    return best_dir


def choose_primary_exec(exec_files: list[Path], report_dir: Path | None) -> Path | None:
    if report_dir is not None:
        maven_exec = report_dir.parent.parent / "jacoco.exec"
        if maven_exec.exists():
            return maven_exec.resolve()
        gradle_exec = report_dir.parent / "jacoco.exec"
        if gradle_exec.exists():
            return gradle_exec.resolve()
    return exec_files[-1] if exec_files else None


def coverage_percent(covered: int, missed: int) -> float:
    total = covered + missed
    if total == 0:
        return 100.0
    return round(covered * 100.0 / total, 2)


def parse_counter_map(xml_path: Path) -> dict[str, dict[str, int]]:
    root = ET.parse(xml_path).getroot()
    counters: dict[str, dict[str, int]] = {}
    for counter in root.findall("counter"):
        counter_type = counter.get("type", "")
        counters[counter_type] = {
            "missed": int(counter.get("missed", "0")),
            "covered": int(counter.get("covered", "0")),
        }
    return counters


def counters_to_metrics_rows(counters: dict[str, dict[str, int]]) -> list[dict[str, Any]]:
    label_map = {
        "INSTRUCTION": "Instruction",
        "BRANCH": "Branch",
        "LINE": "Line",
        "METHOD": "Method",
        "CLASS": "Class",
        "COMPLEXITY": "Complexity",
    }
    rows: list[dict[str, Any]] = []
    for counter_type in COUNTER_TYPES:
        values = counters.get(counter_type, {"missed": 0, "covered": 0})
        covered = values.get("covered", 0)
        missed = values.get("missed", 0)
        rows.append(
            {
                "metric_name": f"{label_map[counter_type]} Covered",
                "covered": covered,
                "missed": missed,
                "coverage_percent": coverage_percent(covered, missed),
            }
        )
    return rows


def element_counters(element: ET.Element) -> dict[str, dict[str, int]]:
    counters: dict[str, dict[str, int]] = {}
    for counter in element.findall("counter"):
        counter_type = counter.get("type", "")
        counters[counter_type] = {
            "missed": int(counter.get("missed", "0")),
            "covered": int(counter.get("covered", "0")),
        }
    return counters


def build_package_summary_rows(xml_path: Path) -> list[dict[str, Any]]:
    root = ET.parse(xml_path).getroot()
    rows: list[dict[str, Any]] = []
    for package in root.findall("package"):
        package_name = package.get("name", "").replace("/", ".")
        counters = element_counters(package)
        rows.append(
            {
                "package": package_name,
                "instruction_coverage": coverage_percent(
                    counters.get("INSTRUCTION", {}).get("covered", 0),
                    counters.get("INSTRUCTION", {}).get("missed", 0),
                ),
                "branch_coverage": coverage_percent(
                    counters.get("BRANCH", {}).get("covered", 0),
                    counters.get("BRANCH", {}).get("missed", 0),
                ),
                "line_coverage": coverage_percent(
                    counters.get("LINE", {}).get("covered", 0),
                    counters.get("LINE", {}).get("missed", 0),
                ),
                "method_coverage": coverage_percent(
                    counters.get("METHOD", {}).get("covered", 0),
                    counters.get("METHOD", {}).get("missed", 0),
                ),
                "class_coverage": coverage_percent(
                    counters.get("CLASS", {}).get("covered", 0),
                    counters.get("CLASS", {}).get("missed", 0),
                ),
                "complexity_coverage": coverage_percent(
                    counters.get("COMPLEXITY", {}).get("covered", 0),
                    counters.get("COMPLEXITY", {}).get("missed", 0),
                ),
            }
        )
    return rows


def build_class_summary_rows(xml_path: Path) -> list[dict[str, Any]]:
    root = ET.parse(xml_path).getroot()
    rows: list[dict[str, Any]] = []
    for package in root.findall("package"):
        package_name = package.get("name", "").replace("/", ".")
        for class_el in package.findall("class"):
            class_name = class_el.get("name", "").split("/")[-1]
            counters = element_counters(class_el)
            rows.append(
                {
                    "package": package_name,
                    "class": class_name,
                    "instruction_coverage": coverage_percent(
                        counters.get("INSTRUCTION", {}).get("covered", 0),
                        counters.get("INSTRUCTION", {}).get("missed", 0),
                    ),
                    "branch_coverage": coverage_percent(
                        counters.get("BRANCH", {}).get("covered", 0),
                        counters.get("BRANCH", {}).get("missed", 0),
                    ),
                    "line_coverage": coverage_percent(
                        counters.get("LINE", {}).get("covered", 0),
                        counters.get("LINE", {}).get("missed", 0),
                    ),
                    "method_coverage": coverage_percent(
                        counters.get("METHOD", {}).get("covered", 0),
                        counters.get("METHOD", {}).get("missed", 0),
                    ),
                    "complexity_coverage": coverage_percent(
                        counters.get("COMPLEXITY", {}).get("covered", 0),
                        counters.get("COMPLEXITY", {}).get("missed", 0),
                    ),
                }
            )
    return rows


def build_repository_metrics_row(
    xml_path: Path,
    repo_stats: dict[str, Any],
    total_execution_time: float,
) -> dict[str, Any]:
    root = ET.parse(xml_path).getroot()
    counters = parse_counter_map(xml_path)
    packages = root.findall("package")
    classes = root.findall(".//class")
    return {
        "Total Packages": len(packages),
        "Total Classes": len(classes),
        "Instruction Coverage %": coverage_percent(
            counters.get("INSTRUCTION", {}).get("covered", 0),
            counters.get("INSTRUCTION", {}).get("missed", 0),
        ),
        "Branch Coverage %": coverage_percent(
            counters.get("BRANCH", {}).get("covered", 0),
            counters.get("BRANCH", {}).get("missed", 0),
        ),
        "Line Coverage %": coverage_percent(
            counters.get("LINE", {}).get("covered", 0),
            counters.get("LINE", {}).get("missed", 0),
        ),
        "Method Coverage %": coverage_percent(
            counters.get("METHOD", {}).get("covered", 0),
            counters.get("METHOD", {}).get("missed", 0),
        ),
        "Class Coverage %": coverage_percent(
            counters.get("CLASS", {}).get("covered", 0),
            counters.get("CLASS", {}).get("missed", 0),
        ),
        "Complexity Coverage %": coverage_percent(
            counters.get("COMPLEXITY", {}).get("covered", 0),
            counters.get("COMPLEXITY", {}).get("missed", 0),
        ),
        "Execution Time (seconds)": total_execution_time,
        "Repository Name": repo_stats["repository_name"],
        "Build Tool": repo_stats["build_tool"],
    }


def copy_artifact(source: Path | None, destination: Path) -> bool:
    if source is None or not source.exists():
        return False
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    return True


def execute_build_and_jacoco(
    repo_path: Path,
    build_tool: str,
    env: dict[str, str],
    logger: NotebookLogger,
) -> tuple[BuildStatus, str]:
    chunks: list[str] = []
    status = BuildStatus(build_tool=build_tool, build_command=[], jacoco_command=[])

    if build_tool == "Maven":
        maven = resolve_maven_command(repo_path, logger)
        status.build_command = [*maven, "clean", "test"]
        status.jacoco_command = [*maven, "clean", "test", "jacoco:report"]
    else:
        gradle = resolve_gradle_command(repo_path, logger)
        status.build_command = [*gradle, "clean", "test"]
        status.jacoco_command = [*gradle, "clean", "test", "jacocoTestReport"]

    build_result = run_shell_command(status.build_command, repo_path, env, logger, "build")
    status.build_result = build_result
    status.build_success = build_result.exit_code == 0
    status.test_success = "BUILD SUCCESS" in combine_streams(build_result.stdout, build_result.stderr) or build_result.exit_code == 0
    chunks.append(f"===== {' '.join(status.build_command)} =====")
    chunks.append(combine_streams(build_result.stdout, build_result.stderr))

    report_dirs = find_jacoco_report_dirs(repo_path)
    if not report_dirs and build_tool == "Maven":
        for module_pom in repo_path.rglob("pom.xml"):
            module_dir = module_pom.parent
            if module_dir == repo_path:
                continue
            module_text = module_pom.read_text(encoding="utf-8", errors="replace")
            if "jacoco-maven-plugin" not in module_text:
                continue
            jacoco_result = run_shell_command(
                [*resolve_maven_command(repo_path, logger), "jacoco:report"],
                module_dir,
                env,
                logger,
                "jacoco",
            )
            status.jacoco_result = jacoco_result
            chunks.append(f"===== module jacoco report {module_dir} =====")
            chunks.append(combine_streams(jacoco_result.stdout, jacoco_result.stderr))
        report_dirs = find_jacoco_report_dirs(repo_path)
        status.test_success = status.test_success or (status.jacoco_result.exit_code == 0 if status.jacoco_result else False)

    exec_files = find_jacoco_exec_files(repo_path)
    status.report_dir = choose_primary_report_dir(report_dirs)
    status.jacoco_exec = choose_primary_exec(exec_files, status.report_dir)
    if status.report_dir is not None:
        status.jacoco_xml = status.report_dir / "jacoco.xml"
        status.jacoco_csv = status.report_dir / "jacoco.csv"
        status.index_html = status.report_dir / "index.html"
        status.report_generated = status.jacoco_xml.exists()
    if not status.report_generated:
        logger.error("JaCoCo report files were not found after build.", step="jacoco")

    return status, "\n".join(chunks)


def collect_outputs(
    status: BuildStatus,
    repo_stats: dict[str, Any],
    output_dir: Path,
    total_execution_time: float,
    logger: NotebookLogger,
) -> dict[str, Any]:
    ensure_output_dir(output_dir)
    copied = {
        "jacoco.exec": copy_artifact(status.jacoco_exec, output_dir / "jacoco.exec"),
        "jacoco.xml": copy_artifact(status.jacoco_xml, output_dir / "jacoco.xml"),
        "jacoco.csv": copy_artifact(status.jacoco_csv, output_dir / "jacoco.csv"),
        "index.html": copy_artifact(status.index_html, output_dir / "index.html"),
    }
    if not copied["jacoco.xml"]:
        raise FileNotFoundError("Unable to copy jacoco.xml to outputs.")

    xml_path = output_dir / "jacoco.xml"
    metrics_df = pd.DataFrame(counters_to_metrics_rows(parse_counter_map(xml_path)))
    metrics_df.to_csv(output_dir / "jacoco_metrics.csv", index=False)

    package_df = pd.DataFrame(build_package_summary_rows(xml_path))
    package_df.to_csv(output_dir / "package_summary.csv", index=False)

    class_df = pd.DataFrame(build_class_summary_rows(xml_path))
    class_df.to_csv(output_dir / "class_summary.csv", index=False)

    repository_metrics = build_repository_metrics_row(xml_path, repo_stats, total_execution_time)
    pd.DataFrame([repository_metrics]).to_csv(output_dir / "repository_metrics.csv", index=False)

    return {
        "copied": copied,
        "metrics_df": metrics_df,
        "package_df": package_df,
        "class_df": class_df,
        "repository_metrics": repository_metrics,
    }


def preview_raw_output(raw_text: str, preview_lines: int, output_path: Path) -> None:
    lines = raw_text.splitlines()
    print(f"\n{'=' * 80}")
    print(f"RAW JACOCO CONSOLE OUTPUT PREVIEW (first {preview_lines} lines)")
    print(f"{'=' * 80}\n")
    if not lines:
        print("(empty raw output)")
        return
    print("\n".join(lines[:preview_lines]))
    remaining = len(lines) - preview_lines
    if remaining > 0:
        print(f"\n... ({remaining} more lines saved to {output_path})")


"""Path coverage evidence analysis helpers for JaCoCo artifacts."""
from __future__ import annotations

import csv
import re
import sys
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import Any

import pandas as pd

TOOL_ROOT = Path(__file__).resolve().parent
JACOCO_TOOL_ROOT = TOOL_ROOT.parent.parent / "JaCoCo Coverage" / "tool"
if str(JACOCO_TOOL_ROOT) not in sys.path:
    sys.path.insert(0, str(JACOCO_TOOL_ROOT))

from _jacoco_notebook_utils import (  # noqa: E402
    BuildStatus,
    copy_artifact,
    ensure_output_dir,
)

PATH_KEYWORDS = [
    "PATH",
    "Execution Path",
    "Path Coverage",
    "Traversal",
    "Loop Path",
    "Nested Path",
    "Exception Path",
    "Call Path",
    "Call Graph",
    "Function Path",
    "Route",
    "CFG",
    "Control Flow Graph",
    "Path ID",
    "Execution Trace",
    "Visited Path",
    "Unique Path",
]

PATH_METRICS = [
    "Path Execution Tracking",
    "Complete Coverage Path Verification",
    "Partial Path Coverage Detection",
    "Nested Condition Path Testing",
    "Loop Path Detection",
    "Unreachable Path Detection",
    "Exception Path Handling",
    "Multi-Function Path Tracking",
    "CI/CD Integration Test",
    "Path Detection Testing",
]

METRIC_KEYWORD_MAP: dict[str, list[str]] = {
    "Path Execution Tracking": ["path execution", "execution path", "execution trace", "visited path", "path id"],
    "Complete Coverage Path Verification": ["path coverage", "complete coverage path", "unique path"],
    "Partial Path Coverage Detection": ["partial path", "path coverage"],
    "Nested Condition Path Testing": ["nested path", "nested condition path"],
    "Loop Path Detection": ["loop path"],
    "Unreachable Path Detection": ["unreachable path"],
    "Exception Path Handling": ["exception path"],
    "Multi-Function Path Tracking": ["function path", "call path", "call graph", "multi-function"],
    "CI/CD Integration Test": ["ci/cd", "cicd integration"],
    "Path Detection Testing": ["path detection", "path coverage", "control flow graph", "cfg"],
}

JACOCO_COUNTER_TYPES = {"INSTRUCTION", "BRANCH", "LINE", "METHOD", "CLASS", "COMPLEXITY"}
IDENTIFIER_ATTRS = {"name", "sourcefilename", "desc", "class", "method"}


def copy_raw_jacoco_artifacts(status: BuildStatus, output_dir: Path) -> dict[str, bool]:
    ensure_output_dir(output_dir)
    return {
        "jacoco.exec": copy_artifact(status.jacoco_exec, output_dir / "jacoco.exec"),
        "jacoco.xml": copy_artifact(status.jacoco_xml, output_dir / "jacoco.xml"),
        "jacoco.csv": copy_artifact(status.jacoco_csv, output_dir / "jacoco.csv"),
        "index.html": copy_artifact(status.index_html, output_dir / "index.html"),
    }


def element_path(parent_path: str, element: ET.Element) -> str:
    tag = element.tag.split("}")[-1] if "}" in element.tag else element.tag
    return f"{parent_path}/{tag}" if parent_path else tag


def dump_jacoco_xml_nodes(xml_path: Path, output_csv: Path) -> pd.DataFrame:
    root = ET.parse(xml_path).getroot()
    rows: list[dict[str, Any]] = []

    def walk(element: ET.Element, parent: str, depth: int) -> None:
        path = element_path(parent, element)
        attributes = "; ".join(f"{key}={value}" for key, value in sorted(element.attrib.items()))
        text = (element.text or "").strip()
        rows.append(
            {
                "element_path": path,
                "tag": element.tag.split("}")[-1] if "}" in element.tag else element.tag,
                "depth": depth,
                "attributes": attributes,
                "text": text,
            }
        )
        for counter in element.findall("counter"):
            counter_type = counter.get("type", "")
            rows.append(
                {
                    "element_path": f"{path}/counter[@type={counter_type}]",
                    "tag": "counter",
                    "depth": depth + 1,
                    "attributes": f"type={counter_type}; missed={counter.get('missed', '')}; covered={counter.get('covered', '')}",
                    "text": "",
                }
            )
        for child in element:
            if child.tag.endswith("counter"):
                continue
            walk(child, path, depth + 1)

    walk(root, "", 0)
    frame = pd.DataFrame(rows)
    frame.to_csv(output_csv, index=False)
    return frame


def extract_xml_counter_types(xml_path: Path) -> set[str]:
    root = ET.parse(xml_path).getroot()
    return {counter.get("type", "") for counter in root.iter() if counter.tag.endswith("counter")}


def read_artifact_text(path: Path | None) -> str:
    if path is None or not path.exists():
        return ""
    if path.suffix.lower() == ".exec":
        try:
            raw = path.read_bytes()
            return raw.decode("utf-8", errors="ignore")
        except OSError:
            return ""
    return path.read_text(encoding="utf-8", errors="replace")


def find_keyword_matches(text: str, keyword: str) -> list[str]:
    if not text:
        return []
    pattern = re.compile(re.escape(keyword), re.IGNORECASE)
    matches: list[str] = []
    for line in text.splitlines():
        if pattern.search(line):
            matches.append(line.strip()[:300])
    if not matches and pattern.search(text):
        start = pattern.search(text).start()
        matches.append(text[max(0, start - 40) : start + 80].replace("\n", " ")[:300])
    return matches


def is_schema_level_evidence(keyword: str, matched_text: str, artifact: str) -> bool:
    lowered = matched_text.lower()
    keyword_lower = keyword.lower()
    if artifact == "jacoco.xml":
        if 'type="PATH"' in matched_text or "type=PATH" in matched_text:
            return True
        if keyword_lower == "path coverage" and "path coverage" in lowered and "counter" in lowered:
            return True
        if keyword_lower in {"cfg", "control flow graph"} and "control flow graph" in lowered:
            return True
        if keyword_lower == "route" and ('name="route"' in matched_text or "name=route" in matched_text):
            return False
        if any(attr in matched_text for attr in ('name="', "sourcefilename=", "desc=")):
            return False
        if "<counter" in matched_text and keyword_lower == "path":
            return 'type="PATH"' in matched_text or "type=PATH" in matched_text
    if artifact == "jacoco.csv":
        return keyword_lower in lowered.split(",")[0:1] or keyword_lower in lowered[:120]
    if artifact == "index.html":
        return keyword_lower in {"path coverage", "execution path", "control flow graph", "cfg"} and keyword_lower in lowered
    if artifact == "jacoco_console_output.txt":
        return keyword_lower in {"path coverage", "execution path", "jacoco"} and keyword_lower in lowered
    return False


def search_path_keywords(artifacts: dict[str, Path | None]) -> pd.DataFrame:
    rows: list[dict[str, str]] = []
    for artifact_name, artifact_path in artifacts.items():
        text = read_artifact_text(artifact_path)
        for keyword in PATH_KEYWORDS:
            matches = find_keyword_matches(text, keyword)
            rows.append(
                {
                    "Artifact": artifact_name,
                    "Keyword": keyword,
                    "Found (Yes/No)": "Yes" if matches else "No",
                    "Matched Text": matches[0] if matches else "",
                }
            )
    return pd.DataFrame(rows, columns=["Artifact", "Keyword", "Found (Yes/No)", "Matched Text"])


def validate_path_metrics(
    keyword_df: pd.DataFrame,
    artifacts: dict[str, Path | None],
    xml_path: Path | None,
) -> pd.DataFrame:
    counter_types = extract_xml_counter_types(xml_path) if xml_path and xml_path.exists() else set()
    rows: list[dict[str, str]] = []

    for metric in PATH_METRICS:
        keywords = METRIC_KEYWORD_MAP.get(metric, [])
        supported = False
        evidence_source = ""
        artifact_name = ""
        comments = ""

        if "PATH" in counter_types or any("PATH" in value for value in counter_types):
            supported = True
            evidence_source = "JaCoCo XML counter type"
            artifact_name = "jacoco.xml"
            comments = f"Found counter types: {sorted(counter_types)}"

        if not supported:
            for keyword in keywords:
                hits = keyword_df[
                    (keyword_df["Keyword"].str.lower() == keyword.lower())
                    & (keyword_df["Found (Yes/No)"] == "Yes")
                ]
                for _, hit in hits.iterrows():
                    if is_schema_level_evidence(keyword, str(hit["Matched Text"]), str(hit["Artifact"])):
                        supported = True
                        evidence_source = "Keyword match in JaCoCo artifact schema/report text"
                        artifact_name = str(hit["Artifact"])
                        comments = str(hit["Matched Text"])[:300]
                        break
                if supported:
                    break

        if supported:
            status = "Supported"
        elif xml_path and xml_path.exists():
            status = "Not Supported"
            comments = (
                comments
                or f"JaCoCo emits counters {sorted(counter_types)} only; no explicit Path Coverage metric found."
            )
        else:
            status = "No Evidence Found"
            comments = comments or "Required JaCoCo artifacts were not available for validation."

        rows.append(
            {
                "Metric": metric,
                "Supported": status,
                "Evidence Source": evidence_source,
                "Artifact": artifact_name,
                "Comments": comments,
            }
        )
    return pd.DataFrame(rows, columns=["Metric", "Supported", "Evidence Source", "Artifact", "Comments"])


"""Static DU raw output extraction helpers."""
from __future__ import annotations

import json
import re
import shutil
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import pandas as pd

TOOL_ROOT = Path(__file__).resolve().parent
JACOCO_TOOL_ROOT = TOOL_ROOT.parent.parent / "JaCoCo Coverage" / "tool"
if str(JACOCO_TOOL_ROOT) not in sys.path:
    sys.path.insert(0, str(JACOCO_TOOL_ROOT))

from _jacoco_notebook_utils import (  # noqa: E402
    CommandResult,
    NotebookLogger,
    combine_streams,
    compute_repository_stats,
    detect_build_tool,
    discover_java_files,
    ensure_output_dir,
    extract_package_name,
    resolve_gradle_command,
    resolve_maven_command,
    run_shell_command,
    save_java_inventory,
)

DATA_FLOW_METRICS = [
    "Variable Definition Detection",
    "Definition-Use Mapping",
    "Coverage Measurement",
    "Uncovered Definition Detection",
    "Variable Use Detection",
]

METRIC_EVIDENCE_KEYS: dict[str, list[str]] = {
    "Variable Definition Detection": [
        "definitions_total",
        "definitions_covered",
        "all_defs_percent",
        "all_defs_coverage_score",
    ],
    "Definition-Use Mapping": [
        "du_paths",
        "du_pairs_total",
        "du_pairs_covered",
        "data_path_correlation_percent",
        "data_path_correlation_score",
    ],
    "Coverage Measurement": [
        "du_path_percent",
        "all_uses_percent",
        "all_defs_percent",
        "du_path_validation_score",
    ],
    "Uncovered Definition Detection": [
        "uncovered_definitions",
        "dead_data_identification_score",
    ],
    "Variable Use Detection": [
        "uses_total",
        "uses_covered",
        "c_use_total",
        "p_use_total",
        "all_uses_percent",
        "all_uses_coverage_score",
    ],
}

DERIVED_SCORE_KEYS = {
    "dead_data_identification_score",
    "all_defs_coverage_score",
    "data_path_correlation_score",
    "du_path_validation_score",
    "all_uses_coverage_score",
}

STATIC_DU_MAIN_CLASS = "com.testable.training.platform.StaticDuTrigger"
STATIC_DU_PLATFORM_DIR = "static-du-platform"

USE_TYPE_MAP = {
    "computational": "C-Use",
    "return": "C-Use",
    "predicate": "P-Use",
    "c-use": "C-Use",
    "p-use": "P-Use",
}

CLASS_NAME_RE = re.compile(r"(?:public\s+)?(?:final\s+)?class\s+(\w+)")
METHOD_RE = re.compile(r"(?:public|private|protected)\s+[\w<>,\s\[\]]+\s+(\w+)\s*\(")


@dataclass
class StaticDuRunStatus:
    command: list[str]
    build_command: list[str]
    build_result: CommandResult | None
    run_result: CommandResult | None
    build_success: bool = False
    run_success: bool = False
    static_du_json: Path | None = None
    static_du_summary_json: Path | None = None
    du_path_correlation_json: Path | None = None
    static_du_meta_json: Path | None = None


def detect_static_du_platform(repo_path: Path) -> Path | None:
    platform = repo_path / STATIC_DU_PLATFORM_DIR
    if (platform / "pom.xml").exists():
        return platform.resolve()
    for pom in repo_path.rglob("pom.xml"):
        if pom.parent.name == STATIC_DU_PLATFORM_DIR:
            return pom.parent.resolve()
    return None


def resolve_static_du_command(repo_path: Path, logger: NotebookLogger, skip_verify: bool = False) -> list[str]:
    platform = detect_static_du_platform(repo_path)
    if platform is None:
        raise FileNotFoundError(
            f"No {STATIC_DU_PLATFORM_DIR} module found in repository. Cannot execute Static DU trigger."
        )
    maven = resolve_maven_command(repo_path, logger)
    command = [
        *maven,
        "-pl",
        STATIC_DU_PLATFORM_DIR,
        "exec:java",
        f"-Dexec.mainClass={STATIC_DU_MAIN_CLASS}",
    ]
    if skip_verify:
        command.append("-Dexec.args=--skip-verify")
    return command


def execute_build_only(
    repo_path: Path,
    build_tool: str,
    env: dict[str, str],
    logger: NotebookLogger,
) -> CommandResult:
    if build_tool == "Maven":
        command = [*resolve_maven_command(repo_path, logger), "clean", "compile"]
        platform = detect_static_du_platform(repo_path)
        if platform is not None:
            command = [*resolve_maven_command(repo_path, logger), "clean", "compile", "-pl", STATIC_DU_PLATFORM_DIR, "-am"]
    else:
        command = [*resolve_gradle_command(repo_path, logger), "clean", "build", "-x", "test"]
    return run_shell_command(command, repo_path, env, logger, "build")


def execute_static_du(
    repo_path: Path,
    env: dict[str, str],
    logger: NotebookLogger,
    skip_verify: bool = True,
) -> tuple[StaticDuRunStatus, str]:
    chunks: list[str] = []
    status = StaticDuRunStatus(command=[], build_command=[], build_result=None, run_result=None)
    build_tool = detect_build_tool(repo_path)

    if build_tool == "Maven":
        build_command = [*resolve_maven_command(repo_path, logger), "clean", "compile"]
        if detect_static_du_platform(repo_path) is not None:
            build_command = [
                *resolve_maven_command(repo_path, logger),
                "clean",
                "compile",
                "-pl",
                STATIC_DU_PLATFORM_DIR,
                "-am",
            ]
    else:
        build_command = [*resolve_gradle_command(repo_path, logger), "clean", "build", "-x", "test"]
    status.build_command = build_command
    build_result = run_shell_command(build_command, repo_path, env, logger, "build")
    status.build_result = build_result
    status.build_success = build_result.exit_code == 0
    chunks.append(f"===== {' '.join(build_result.command)} =====")
    chunks.append(combine_streams(build_result.stdout, build_result.stderr))

    try:
        status.command = resolve_static_du_command(repo_path, logger, skip_verify=skip_verify)
    except FileNotFoundError as exc:
        logger.error(str(exc), step="static_du_detect")
        return status, "\n".join(chunks)

    run_result = run_shell_command(status.command, repo_path, env, logger, "static_du_run")
    status.run_result = run_result
    status.run_success = run_result.exit_code == 0
    chunks.append(f"===== {' '.join(status.command)} =====")
    chunks.append(combine_streams(run_result.stdout, run_result.stderr))

    status.static_du_json = repo_path / "static_du.json"
    status.static_du_summary_json = repo_path / "artifacts" / "training" / "static_du_summary.json"
    status.du_path_correlation_json = repo_path / "artifacts" / "training" / "du_path_correlation.json"
    status.static_du_meta_json = repo_path / "artifacts" / "training" / "static_du_meta.json"
    return status, "\n".join(chunks)


def copy_artifact(source: Path | None, destination: Path) -> bool:
    if source is None or not source.exists():
        return False
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    return True


def preserve_static_du_artifacts(status: StaticDuRunStatus, output_dir: Path, repo_path: Path | None = None) -> dict[str, bool]:
    ensure_output_dir(output_dir)
    copied = {
        "static_du_output.json": copy_artifact(status.static_du_json, output_dir / "static_du_output.json"),
        "static_du_summary": copy_artifact(
            status.static_du_summary_json,
            output_dir / "static_du_summary_copy.json",
        ),
    }
    if repo_path is not None:
        for pattern in ("static_du*.xml", "static_du*.csv", "*static_du*.xml", "*static_du*.csv"):
            for source in repo_path.rglob(pattern.split("/")[-1]):
                if not source.is_file():
                    continue
                suffix = source.suffix.lower()
                if suffix == ".xml":
                    copied["static_du_output.xml"] = copy_artifact(source, output_dir / "static_du_output.xml")
                elif suffix == ".csv":
                    copied["static_du_output.csv"] = copy_artifact(source, output_dir / "static_du_output.csv")
    return copied


def load_json_map(path: Path) -> dict[str, Any]:
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


def merged_summary_payload(repo_path: Path, output_dir: Path) -> dict[str, Any]:
    candidates = [
        output_dir / "static_du_output.json",
        repo_path / "static_du.json",
        repo_path / "artifacts" / "training" / "static_du_summary.json",
        repo_path / "artifacts" / "training" / "du_path_correlation.json",
    ]
    merged: dict[str, Any] = {}
    for path in candidates:
        if not path.exists():
            continue
        payload = load_json_map(path)
        if "supplemental_raw_data" in payload:
            supplemental = payload.get("supplemental_raw_data", {})
            if isinstance(supplemental, dict):
                summary = supplemental.get("static_du_summary", {})
                if isinstance(summary, dict):
                    merged.update(summary)
                correlation = supplemental.get("du_path_correlation", {})
                if isinstance(correlation, dict):
                    for key, value in correlation.items():
                        merged.setdefault(key, value)
        if "summary" in payload and isinstance(payload["summary"], dict):
            merged.update(payload["summary"])
        if "definitions_total" in payload:
            merged.update(payload)
        if "du_paths" in payload and "du_paths" not in merged:
            merged["du_paths"] = payload["du_paths"]
    return merged


def map_use_type(raw_value: str) -> str:
    normalized = raw_value.strip().lower()
    return USE_TYPE_MAP.get(normalized, raw_value)


def extract_class_name(java_path: Path) -> str:
    try:
        text = java_path.read_text(encoding="utf-8")
    except (OSError, UnicodeDecodeError):
        return java_path.stem
    match = CLASS_NAME_RE.search(text)
    return match.group(1) if match else java_path.stem


def find_java_file_by_name(repo_path: Path, file_name: str) -> Path | None:
    for path in repo_path.rglob(file_name):
        if any(part in {".git", "target", "build"} for part in path.parts):
            continue
        return path.resolve()
    return None


def extract_variable_definitions(summary: dict[str, Any], repo_path: Path) -> pd.DataFrame:
    rows: list[dict[str, str]] = []
    definitions = summary.get("definitions")
    if isinstance(definitions, list):
        for item in definitions:
            if not isinstance(item, dict):
                continue
            rows.append(
                {
                    "file": str(item.get("file", "")),
                    "class": str(item.get("class", "")),
                    "method": str(item.get("method", "")),
                    "variable": str(item.get("variable", "")),
                    "definition_line": str(item.get("definition_line", item.get("line", ""))),
                    "definition_type": str(item.get("definition_type", item.get("type", ""))),
                }
            )
    return pd.DataFrame(
        rows,
        columns=["file", "class", "method", "variable", "definition_line", "definition_type"],
    )


def extract_definition_use_pairs(summary: dict[str, Any], repo_path: Path) -> pd.DataFrame:
    rows: list[dict[str, str]] = []
    du_paths = summary.get("du_paths", [])
    if not isinstance(du_paths, list):
        return pd.DataFrame(
            columns=[
                "file",
                "class",
                "method",
                "variable",
                "definition_line",
                "use_line",
                "use_type",
                "du_pair",
            ]
        )

    for item in du_paths:
        if not isinstance(item, dict):
            continue
        file_name = str(item.get("file", ""))
        variable = str(item.get("variable", ""))
        use_line = str(item.get("line", ""))
        emitted_use_type = str(item.get("use_type", ""))
        mapped_use_type = map_use_type(emitted_use_type)
        java_path = find_java_file_by_name(repo_path, file_name) if file_name else None
        class_name = extract_class_name(java_path) if java_path else ""
        du_pair = f"{variable}@{use_line}" if variable and use_line else variable
        rows.append(
            {
                "file": file_name,
                "class": class_name,
                "method": "",
                "variable": variable,
                "definition_line": "",
                "use_line": use_line,
                "use_type": mapped_use_type,
                "du_pair": du_pair,
            }
        )
    return pd.DataFrame(rows)


def collect_metric_evidence(summary: dict[str, Any], unified_json: dict[str, Any]) -> dict[str, list[str]]:
    evidence: dict[str, list[str]] = {metric: [] for metric in DATA_FLOW_METRICS}
    for key, value in summary.items():
        for metric, keys in METRIC_EVIDENCE_KEYS.items():
            if key in keys:
                evidence[metric].append(f"{key}={value}")
    metrics_rows = unified_json.get("metrics", [])
    if isinstance(metrics_rows, list):
        for row in metrics_rows:
            if not isinstance(row, dict):
                continue
            l5 = str(row.get("l5_metric", ""))
            if l5 in evidence:
                for field in ("covered", "score", "value", "formula"):
                    if field in row:
                        evidence[l5].append(f"{field}={row[field]}")
    return evidence


def validate_data_flow_metrics(summary: dict[str, Any], unified_json: dict[str, Any]) -> pd.DataFrame:
    evidence_map = collect_metric_evidence(summary, unified_json)
    rows: list[dict[str, str]] = []
    for metric in DATA_FLOW_METRICS:
        evidence_parts = evidence_map.get(metric, [])
        directly_emitted = any(
            part.split("=", 1)[0] not in DERIVED_SCORE_KEYS
            for part in evidence_parts
            if "=" in part
        )
        derived = any(
            part.split("=", 1)[0] in DERIVED_SCORE_KEYS
            for part in evidence_parts
            if "=" in part
        )
        if evidence_parts:
            supported = "Supported"
            evidence = " | ".join(evidence_parts[:6])
        else:
            supported = "Not Supported"
            evidence = ""
        rows.append(
            {
                "Metric": metric,
                "Supported": supported,
                "Directly Emitted": "Yes" if directly_emitted else "No",
                "Derived": "Yes" if derived and not directly_emitted else ("Yes" if derived else "No"),
                "Evidence": evidence,
            }
        )
    return pd.DataFrame(rows)


def build_repository_metrics(
    java_files: list[Path],
    summary: dict[str, Any],
    du_pairs_df: pd.DataFrame,
    execution_time: float,
) -> pd.DataFrame:
    classes = {extract_class_name(path) for path in java_files}
    c_uses = int((du_pairs_df["use_type"] == "C-Use").sum()) if not du_pairs_df.empty else int(summary.get("c_use_total", 0) or 0)
    p_uses_reported = int(summary.get("p_use_total", 0) or 0)
    p_uses_pairs = int((du_pairs_df["use_type"] == "P-Use").sum()) if not du_pairs_df.empty else 0
    files_emitted = summary.get("files", [])
    total_classes = len(files_emitted) if isinstance(files_emitted, list) and files_emitted else len(classes)
    return pd.DataFrame(
        [
            {
                "Total Java Files": len(java_files),
                "Total Classes": total_classes,
                "Total Methods": "",
                "Total Variable Definitions": int(summary.get("definitions_total", 0) or 0),
                "Total Variable Uses": int(summary.get("uses_total", 0) or 0),
                "Total DU Pairs": int(summary.get("du_pairs_total", 0) or 0),
                "Total C-Uses": c_uses if c_uses else int(summary.get("c_use_total", 0) or 0),
                "Total P-Uses": p_uses_reported if p_uses_reported else p_uses_pairs,
                "Execution Time": execution_time,
            }
        ]
    )


def build_dashboard_table(repo_metrics: pd.DataFrame) -> pd.DataFrame:
    if repo_metrics.empty:
        return pd.DataFrame(columns=["Metric", "Value"])
    row = repo_metrics.iloc[0].to_dict()
    return pd.DataFrame(
        [
            {"Metric": "Java Files", "Value": row.get("Total Java Files", "")},
            {"Metric": "Classes", "Value": row.get("Total Classes", "")},
            {"Metric": "Methods", "Value": row.get("Total Methods", "")},
            {"Metric": "Variable Definitions", "Value": row.get("Total Variable Definitions", "")},
            {"Metric": "Variable Uses", "Value": row.get("Total Variable Uses", "")},
            {"Metric": "Definition-Use Pairs", "Value": row.get("Total DU Pairs", "")},
            {"Metric": "C-Uses", "Value": row.get("Total C-Uses", "")},
            {"Metric": "P-Uses", "Value": row.get("Total P-Uses", "")},
        ]
    )


def preview_raw_output(raw_text: str, max_lines: int, source_path: Path) -> None:
    lines = raw_text.splitlines()
    print(f"Saved raw output: {source_path} ({len(lines)} lines)")
    preview = "\n".join(lines[:max_lines])
    print(preview)
    if len(lines) > max_lines:
        print(f"... truncated preview ({len(lines) - max_lines} additional lines in file)")


def collect_outputs(
    status: StaticDuRunStatus,
    repo_path: Path,
    java_files: list[Path],
    output_dir: Path,
    execution_time: float,
) -> dict[str, Any]:
    ensure_output_dir(output_dir)
    unified_json = load_json_map(status.static_du_json) if status.static_du_json and status.static_du_json.exists() else {}
    summary = merged_summary_payload(repo_path, output_dir)

    definitions_df = extract_variable_definitions(summary, repo_path)
    du_pairs_df = extract_definition_use_pairs(summary, repo_path)
    metrics_df = validate_data_flow_metrics(summary, unified_json)
    repository_metrics_df = build_repository_metrics(java_files, summary, du_pairs_df, execution_time)
    dashboard_df = build_dashboard_table(repository_metrics_df)

    definitions_df.to_csv(output_dir / "variable_definitions.csv", index=False)
    du_pairs_df.to_csv(output_dir / "definition_use_pairs.csv", index=False)
    metrics_df.to_csv(output_dir / "data_flow_metrics.csv", index=False)
    repository_metrics_df.to_csv(output_dir / "repository_metrics.csv", index=False)
    dashboard_df.to_csv(output_dir / "dashboard.csv", index=False)

    return {
        "definitions_df": definitions_df,
        "du_pairs_df": du_pairs_df,
        "metrics_df": metrics_df,
        "repository_metrics_df": repository_metrics_df,
        "dashboard_df": dashboard_df,
        "summary": summary,
    }


"""JaCoCo + Static DU combined validation helpers."""
from __future__ import annotations

import json
import shutil
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import pandas as pd

TOOL_ROOT = Path(__file__).resolve().parent
JACOCO_TOOL_ROOT = TOOL_ROOT.parent.parent / "JaCoCo Coverage" / "tool"
JACOCO_PATH_TOOL_ROOT = TOOL_ROOT.parent.parent / "JaCoCo Path Analysis" / "tool"
STATIC_DU_TOOL_ROOT = TOOL_ROOT.parent.parent / "Static DU Analysis" / "tool"
for path in (str(JACOCO_TOOL_ROOT), str(JACOCO_PATH_TOOL_ROOT), str(STATIC_DU_TOOL_ROOT), str(TOOL_ROOT)):
    if path not in sys.path:
        sys.path.insert(0, path)

from _jacoco_notebook_utils import (  # noqa: E402
    BuildStatus,
    NotebookLogger,
    combine_streams,
    compute_repository_stats,
    configure_java_runtime,
    copy_artifact,
    counters_to_metrics_rows,
    coverage_percent,
    detect_build_tool,
    discover_java_files,
    ensure_output_dir,
    execute_build_and_jacoco,
    extract_package_name,
    parse_counter_map,
    resolve_maven_command,
    run_shell_command,
    save_java_inventory,
)
from _jacoco_path_analysis_utils import (  # noqa: E402
    JACOCO_COUNTER_TYPES,
    PATH_METRICS,
    copy_raw_jacoco_artifacts,
    extract_xml_counter_types,
    search_path_keywords,
    validate_path_metrics,
)
from _static_du_notebook_utils import (  # noqa: E402
    STATIC_DU_MAIN_CLASS,
    STATIC_DU_PLATFORM_DIR,
)

JACOCO_TRIGGER_CLASS = "com.testable.training.platform.JacocoTrigger"
JACOCO_PLATFORM_DIR = "jacoco-platform"
DEF_USE_TRIGGER_CLASS = "com.testable.training.defuse.DefUseTrigger"
DEF_USE_PLATFORM_DIR = "def-use-platform"

PLATFORM_JSON_ARTIFACTS = [
    "jacoco.json",
    "static_du.json",
    "def_use.json",
    "metrics.json",
    "platform_metrics.json",
    "dashboard_metrics.json",
    "jacoco_metrics.json",
    "static_du_metrics.json",
]

CONTROL_FLOW_METRICS = PATH_METRICS

COVERAGE_REGRESSION_METRICS = [
    "Coverage Delta",
    "Regression Testing Monitoring",
    "Test Suite Effectiveness Tracking",
    "CI/CD Quality Gate Enforcement",
    "Change Impact Analysis",
    "Quality Improvement Measurement",
]

ALL_DEFINITION_METRICS = [
    "Variable Definition Detection",
    "Definition-Use Mapping",
    "Coverage Measurement",
    "Uncovered Definition Detection",
    "Edge Case Handling",
    "Reporting Validation",
]

ALL_USES_METRICS = [
    "Computational Use Detection (C-Use)",
    "Predicate Use Detection (P-Use)",
    "Definition-Use Pair Identification",
    "All-Uses Coverage Verification",
    "Partial Uses Coverage Detection",
    "Multiple Definitions Handling",
    "Cross-Function Use Detection",
    "Unreachable Use Detection",
    "Coverage Reporting Validation",
    "Variable Use Detection",
]

DATA_FLOW_METRICS: list[tuple[str, str, str]] = [
    ("Data Flow Testing", "All Definition Coverage", metric) for metric in ALL_DEFINITION_METRICS
] + [
    ("Data Flow Testing", "All Uses Coverage", metric) for metric in ALL_USES_METRICS
]

DEF_USE_SUMMARY_KEYS = {
    "definitions_total": ["definitionsTotal", "definitions_total"],
    "definitions_covered": ["definitionsCovered", "definitions_covered"],
    "uses_total": ["usesTotal", "uses_total"],
    "uses_covered": ["usesCovered", "uses_covered"],
    "c_use_total": ["cUseTotal", "c_use_total"],
    "p_use_total": ["pUseTotal", "p_use_total"],
    "du_pairs_total": ["duPairsTotal", "du_pairs_total"],
    "du_pairs_covered": ["duPairsCovered", "du_pairs_covered"],
    "uncovered_definitions": ["uncoveredDefinitions", "uncovered_definitions"],
    "partial_uses": ["partialUses", "partial_uses"],
    "ghost_uses": ["ghostUses", "ghost_uses"],
    "multiple_definition_sites": ["multipleDefinitionSites", "multiple_definition_sites"],
    "cross_function_uses": ["crossFunctionUses", "cross_function_uses"],
}

STATIC_DU_DUPLICATION_KEYS = [
    "total_lines",
    "duplicated_lines",
    "duplicated_lines_percent",
    "duplicated_blocks",
    "duplicated_files",
    "duplication_density_percent",
]

# Platform score aliases documented in jacoco-platform/JacocoDashboardMetrics.java
PLATFORM_PROXY_MAP: dict[str, str] = {
    "Path Execution Tracking": "branch_percent",
    "Complete Coverage Path Verification": "path_coverage_percent (= branch_percent)",
    "Partial Path Coverage Detection": "partial_branch_lines heuristic",
    "Nested Condition Path Testing": "branch_percent",
    "Loop Path Detection": "branch_percent",
    "Unreachable Path Detection": "ghost_lines heuristic",
    "Exception Path Handling": "branch_percent",
    "Multi-Function Path Tracking": "line_percent",
    "CI/CD Integration Test": "line_percent >= 80 gate",
    "Path Detection Testing": "path_coverage_percent (= branch_percent)",
    "Coverage Delta": "coverage_delta_percent (platform summary)",
    "Regression Testing Monitoring": "modules_tested / modules_with_churn",
    "Test Suite Effectiveness Tracking": "line_percent, branch_percent, instruction_percent",
    "CI/CD Quality Gate Enforcement": "metric_coverage_complete / metrics_covered",
    "Change Impact Analysis": "modules_with_churn, coverage_delta_percent",
    "Quality Improvement Measurement": "coverage_delta_percent, line_percent",
    "Variable Definition Detection": "all_defs_percent from jacoco-platform StaticDuAnalyzer (regex)",
    "Definition-Use Mapping": "du_path_percent from jacoco-platform StaticDuAnalyzer (regex)",
    "Coverage Measurement": "du_path_percent from jacoco-platform StaticDuAnalyzer (regex)",
    "Uncovered Definition Detection": "uncovered_definitions from jacoco-platform StaticDuAnalyzer (regex)",
    "Edge Case Handling": "branch_percent (platform alias)",
    "Reporting Validation": "metrics_total / metrics_covered (platform metadata)",
    "Computational Use Detection (C-Use)": "c_use_percent from jacoco-platform StaticDuAnalyzer (regex)",
    "Predicate Use Detection (P-Use)": "p_use_percent from jacoco-platform StaticDuAnalyzer (regex)",
    "Definition-Use Pair Identification": "du_path_percent from jacoco-platform StaticDuAnalyzer (regex)",
    "All-Uses Coverage Verification": "all_uses_percent from jacoco-platform StaticDuAnalyzer (regex)",
    "Partial Uses Coverage Detection": "partial_uses heuristic",
    "Multiple Definitions Handling": "multiple_definition_sites heuristic",
    "Cross-Function Use Detection": "line_percent (not inter-procedural)",
    "Unreachable Use Detection": "ghost_uses heuristic",
    "Coverage Reporting Validation": "all_uses_percent (platform alias)",
    "Variable Use Detection": "all_uses_percent from jacoco-platform StaticDuAnalyzer (regex)",
}

REPO_ROUTING_ROWS: list[dict[str, str]] = [
    {
        "Metric_Family": "Native JaCoCo Counters",
        "Taxonomy_Section": "Instruction / Line / Branch / Method / Class / Complexity",
        "Recommended_Pipeline": "JaCoCo Coverage",
        "Recommended_Repo": "java-tool-testing-jacoco",
        "Project_Path": "Java/JaCoCo Coverage",
        "Native_Tool_Output": "jacoco.xml",
        "Notes": "Official JaCoCo counters only; no path or def-use metrics.",
    },
    {
        "Metric_Family": "Control Flow / Path Coverage",
        "Taxonomy_Section": "Control Flow Testing — Path Coverage (10 metrics)",
        "Recommended_Pipeline": "JPF Path Analysis or JaCoCo Path Analysis",
        "Recommended_Repo": "java-tool-testing-jacoco",
        "Project_Path": "Java/JPF Path Analysis",
        "Native_Tool_Output": "jpf path reports / jacoco.xml keywords",
        "Notes": "JaCoCo XML has no PATH counter; def-use repo aliases branch% to path metrics.",
    },
    {
        "Metric_Family": "Coverage Regression / Delta",
        "Taxonomy_Section": "Test Regression/Coverage Analysis (6 metrics)",
        "Recommended_Pipeline": "JaCoCo Coverage with baseline XML comparison",
        "Recommended_Repo": "java-tool-testing-jacoco",
        "Project_Path": "Java/JaCoCo Coverage",
        "Native_Tool_Output": "jacoco.xml baseline vs current delta",
        "Notes": "XML LINE/BRANCH/INSTRUCTION delta is native; churn and quality-gate rows are platform metadata.",
    },
    {
        "Metric_Family": "Data Flow / Definition-Use",
        "Taxonomy_Section": "Data Flow Testing — All Definition + All Uses (16 metrics)",
        "Recommended_Pipeline": "Static DU Analysis",
        "Recommended_Repo": "java-tool-testing-static-du",
        "Project_Path": "Java/Static DU Analysis",
        "Native_Tool_Output": "static_du.json def-use fields",
        "Notes": "java-tool-testing-def-use standalone Static DU emits duplication only; def-use is bundled in jacoco-platform.",
    },
    {
        "Metric_Family": "Combined End-to-End Validation",
        "Taxonomy_Section": "All taxonomy sections (truth-table audit)",
        "Recommended_Pipeline": "JaCoCo Static DU Validation",
        "Recommended_Repo": "java-tool-testing-def-use",
        "Project_Path": "Java/JaCoCo Static DU Validation",
        "Native_Tool_Output": "jacoco.xml + static_du_output.json + jacoco.json platform",
        "Notes": "Use taxonomy_truth_table.csv to separate Native vs Platform_Derived evidence.",
    },
]

TAXONOMY_TRIGGER_MANIFEST: list[dict[str, str]] = [
    {
        "pipeline_id": "jacoco_static_du_validation",
        "pipeline_name": "JaCoCo Static DU Validation (current)",
        "project_path": "Java/JaCoCo Static DU Validation",
        "repo_url": "https://github.com/visvantha-testable/java-tool-testing-def-use.git",
        "repo_folder": "workspace/java-tool-testing-def-use",
        "tools_triggered": "JaCoCo Maven plugin + jacoco-platform JacocoTrigger + static-du-platform StaticDuTrigger",
        "build_command": "mvn clean test",
        "jacoco_trigger": "mvn -pl jacoco-platform exec:java -Dexec.mainClass=com.testable.training.platform.JacocoTrigger -Dexec.args=--skip-verify",
        "static_du_trigger": "mvn -pl static-du-platform exec:java -Dexec.mainClass=com.testable.training.platform.StaticDuTrigger -Dexec.args=--skip-verify",
        "run_command": "python tool/run_jacoco_static_du_validation_benchmark.py",
        "primary_outputs": "outputs/jacoco.xml; outputs/jacoco.json; outputs/static_du_output.json; outputs/taxonomy_truth_table.csv",
        "covers_taxonomy_natively": "1 of 32 metrics (Coverage Delta XML baseline only)",
        "missing_data_to_add": "baseline_jacoco.xml already in artifacts/training; path and def-use need alternate repos below",
    },
    {
        "pipeline_id": "jacoco_coverage",
        "pipeline_name": "JaCoCo Coverage (native counters + baseline delta)",
        "project_path": "Java/JaCoCo Coverage",
        "repo_url": "https://github.com/visvantha-testable/java-tool-testing-jacoco.git",
        "repo_folder": "workspace/java-tool-testing-jacoco",
        "tools_triggered": "JaCoCo Maven plugin",
        "build_command": "mvn clean test",
        "jacoco_trigger": "mvn -pl jacoco-platform exec:java -Dexec.mainClass=com.testable.training.platform.JacocoTrigger -Dexec.args=--skip-verify",
        "static_du_trigger": "",
        "run_command": "cd \"Java/JaCoCo Coverage\" && python tool/run_jacoco_benchmark.py",
        "primary_outputs": "outputs/jacoco.xml; outputs/jacoco.csv; outputs/jacoco_metrics.csv",
        "covers_taxonomy_natively": "INSTRUCTION/LINE/BRANCH/METHOD/CLASS/COMPLEXITY + XML coverage delta",
        "missing_data_to_add": "Clone repo; ensure Maven JaCoCo plugin in pom.xml; optional baseline_jacoco.xml for delta",
    },
    {
        "pipeline_id": "jpf_path_analysis",
        "pipeline_name": "JPF Path Analysis (real path execution)",
        "project_path": "Java/JPF Path Analysis",
        "repo_url": "https://github.com/visvantha-testable/java-tool-testing-jacoco.git",
        "repo_folder": "workspace/java-tool-testing-jacoco",
        "tools_triggered": "Java PathFinder (JPF)",
        "build_command": "mvn clean test (subject repo)",
        "jacoco_trigger": "",
        "static_du_trigger": "",
        "run_command": "cd \"Java/JPF Path Analysis\" && python tool/run_jpf_benchmark.py",
        "primary_outputs": "outputs/jpf_metrics.csv; outputs/path_report.txt; outputs/visited_states.txt",
        "covers_taxonomy_natively": "Control Flow / Path Coverage (10 metrics) via path count, visited states, execution paths",
        "missing_data_to_add": "JPF install at runtimes/jpf-core; .jpf config per class; subject with branching paths",
    },
    {
        "pipeline_id": "static_du_analysis",
        "pipeline_name": "Static DU Analysis (real def-use)",
        "project_path": "Java/Static DU Analysis",
        "repo_url": "https://github.com/visvantha-testable/java-tool-testing-static-du.git",
        "repo_folder": "workspace/java-tool-testing-static-du",
        "tools_triggered": "static-du-platform StaticDuTrigger",
        "build_command": "mvn clean test",
        "jacoco_trigger": "",
        "static_du_trigger": "mvn -pl static-du-platform exec:java -Dexec.mainClass=com.testable.training.platform.StaticDuTrigger -Dexec.args=--skip-verify",
        "run_command": "cd \"Java/Static DU Analysis\" && python tool/run_static_du_benchmark.py",
        "primary_outputs": "outputs/static_du_output.json (definitions_total, c_use_total, p_use_total, du_pairs_total, cross_function_uses)",
        "covers_taxonomy_natively": "Data Flow Testing 16 metrics (all-defs, all-uses, C-use, P-use, DU pairs)",
        "missing_data_to_add": "Clone java-tool-testing-static-du; sample_subject with DataFlowSample.java and test coverage",
    },
]

PIPELINE_RUN_COMMANDS = {
    "Java/JaCoCo Static DU Validation": "python tool/run_jacoco_static_du_validation_benchmark.py",
    "Java/JaCoCo Coverage": "cd \"Java/JaCoCo Coverage\" && python tool/run_jacoco_benchmark.py",
    "Java/JPF Path Analysis": "cd \"Java/JPF Path Analysis\" && python tool/run_jpf_benchmark.py",
    "Java/Static DU Analysis": "cd \"Java/Static DU Analysis\" && python tool/run_static_du_benchmark.py",
}

METRIC_ALTERNATIVE_MAP: dict[str, dict[str, str]] = {
    **{metric: {
        "alternative_tool": "Java PathFinder (JPF)",
        "alternative_repo": "java-tool-testing-jacoco",
        "alternative_pipeline": "Java/JPF Path Analysis",
        "required_data": "JPF .jpf config; runtimes/jpf-core; branching sample code",
        "native_output_fields": "Path Count; Execution Paths; Visited States; search_graph.txt",
    } for metric in CONTROL_FLOW_METRICS},
    **{metric: {
        "alternative_tool": "JaCoCo + baseline XML comparison",
        "alternative_repo": "java-tool-testing-jacoco",
        "alternative_pipeline": "Java/JaCoCo Coverage",
        "required_data": "artifacts/training/baseline_jacoco.xml + current jacoco.xml",
        "native_output_fields": "INSTRUCTION/LINE/BRANCH counter delta in jacoco.xml",
    } for metric in COVERAGE_REGRESSION_METRICS if metric != "Coverage Delta"},
    "Coverage Delta": {
        "alternative_tool": "JaCoCo (already native in current repo)",
        "alternative_repo": "java-tool-testing-def-use",
        "alternative_pipeline": "Java/JaCoCo Static DU Validation",
        "required_data": "baseline_jacoco.xml (already present in artifacts/training/)",
        "native_output_fields": "computed_xml_delta INSTRUCTION/LINE/BRANCH",
    },
    **{metric: {
        "alternative_tool": "Static DU platform (def-use analyzer)",
        "alternative_repo": "java-tool-testing-static-du",
        "alternative_pipeline": "Java/Static DU Analysis",
        "required_data": "static_du.json supplemental_raw_data.static_du_summary with definitions_total, uses_total, c_use_total, p_use_total, du_pairs_total, cross_function_uses",
        "native_output_fields": "definitions_total; uses_total; c_use_total; p_use_total; du_pairs_total; cross_function_uses; du_paths[]",
    } for metric in ALL_DEFINITION_METRICS + ALL_USES_METRICS},
}

TAXONOMY_TRUTH_ROWS: list[tuple[str, str, str, str]] = (
    [("Control Flow Testing", "Path Coverage", metric, "Path Coverage %") for metric in CONTROL_FLOW_METRICS]
    + [
        ("Test Regression/Coverage Analysis", "Coverage Delta", "Coverage Delta", "Coverage Delta %"),
        ("Test Regression/Coverage Analysis", "Coverage Delta", "Regression Testing Monitoring", "Coverage Delta %"),
        (
            "Test Regression/Coverage Analysis",
            "Coverage Delta",
            "Test Suite Effectiveness Tracking",
            "Discovery Power Assessment",
        ),
        (
            "Test Regression/Coverage Analysis",
            "Coverage Delta",
            "CI/CD Quality Gate Enforcement",
            "Deployment Readiness Guard",
        ),
        ("Test Regression/Coverage Analysis", "Coverage Delta", "Change Impact Analysis", "Ripple Effect Mapping"),
        (
            "Test Regression/Coverage Analysis",
            "Coverage Delta",
            "Quality Improvement Measurement",
            "Structural Health Benchmarking",
        ),
    ]
    + [
        ("Data Flow Testing", "All Definition Coverage", metric, goal)
        for metric, goal in [
            ("Variable Definition Detection", "All-Defs Coverage %"),
            ("Definition-Use Mapping", "Data Path Correlation"),
            ("Coverage Measurement", "DU-Path Validation"),
            ("Uncovered Definition Detection", "Dead Data Identification"),
            ("Edge Case Handling", "Null and Boundary Flow Analysis"),
            ("Reporting Validation", "Audit Trail Verification"),
        ]
    ]
    + [
        ("Data Flow Testing", "All Uses Coverage", metric, goal)
        for metric, goal in [
            ("Computational Use Detection (C-Use)", "Data Processing Validation"),
            ("Predicate Use Detection (P-Use)", "Logic Influence Assessment"),
            ("Definition-Use Pair Identification", "Path Correlation Mapping"),
            ("All-Uses Coverage Verification", "Comprehensive Data Proofing"),
            ("Partial Uses Coverage Detection", "Data Flow Gap Analysis"),
            ("Multiple Definitions Handling", "Ambiguity Resolution"),
            ("Cross-Function Use Detection", "Inter-procedural Tracking"),
            ("Unreachable Use Detection", "Ghost Use Identification"),
            ("Coverage Reporting Validation", "Data Integrity Audit"),
            ("Variable Use Detection", "All-Uses Coverage %"),
        ]
    ]
)


@dataclass
class CombinedRunStatus:
    build_status: BuildStatus
    jacoco_trigger_command: list[str]
    static_du_trigger_command: list[str]
    jacoco_trigger_success: bool = False
    static_du_trigger_success: bool = False
    def_use_trigger_success: bool = False
    jacoco_json: Path | None = None
    static_du_json: Path | None = None
    def_use_json: Path | None = None
    unified_trigger: bool = False


def resolve_platform_trigger_command(
    repo_path: Path,
    logger: NotebookLogger,
    platform_dir: str,
    main_class: str,
    skip_verify: bool = True,
) -> list[str]:
    maven = resolve_maven_command(repo_path, logger)
    command = [*maven, "-pl", platform_dir, "exec:java", f"-Dexec.mainClass={main_class}"]
    if skip_verify:
        command.append("-Dexec.args=--skip-verify")
    return command


def execute_platform_triggers(
    repo_path: Path,
    env: dict[str, str],
    logger: NotebookLogger,
    skip_verify: bool = True,
) -> tuple[CombinedRunStatus, str, str, str]:
    build_tool = detect_build_tool(repo_path)
    build_status, jacoco_console = execute_build_and_jacoco(repo_path, build_tool, env, logger)

    unified = (repo_path / DEF_USE_PLATFORM_DIR / "pom.xml").exists()
    if unified:
        trigger_command = resolve_platform_trigger_command(
            repo_path, logger, DEF_USE_PLATFORM_DIR, DEF_USE_TRIGGER_CLASS, skip_verify
        )
        trigger_result = run_shell_command(trigger_command, repo_path, env, logger, "def_use_trigger")
        trigger_console = combine_streams(trigger_result.stdout, trigger_result.stderr)
        success = trigger_result.exit_code == 0
        status = CombinedRunStatus(
            build_status=build_status,
            jacoco_trigger_command=trigger_command,
            static_du_trigger_command=trigger_command,
            jacoco_trigger_success=success,
            static_du_trigger_success=success,
            def_use_trigger_success=success,
            jacoco_json=repo_path / "jacoco.json",
            static_du_json=repo_path / "static_du.json",
            def_use_json=repo_path / "def_use.json",
            unified_trigger=True,
        )
        return status, jacoco_console, trigger_console, trigger_console

    jacoco_trigger_command = resolve_platform_trigger_command(
        repo_path, logger, JACOCO_PLATFORM_DIR, JACOCO_TRIGGER_CLASS, skip_verify
    )
    jacoco_trigger_result = run_shell_command(jacoco_trigger_command, repo_path, env, logger, "jacoco_trigger")
    jacoco_trigger_console = combine_streams(jacoco_trigger_result.stdout, jacoco_trigger_result.stderr)

    static_du_trigger_command = resolve_platform_trigger_command(
        repo_path, logger, STATIC_DU_PLATFORM_DIR, STATIC_DU_MAIN_CLASS, skip_verify
    )
    static_du_trigger_result = run_shell_command(static_du_trigger_command, repo_path, env, logger, "static_du_trigger")
    static_du_trigger_console = combine_streams(static_du_trigger_result.stdout, static_du_trigger_result.stderr)

    status = CombinedRunStatus(
        build_status=build_status,
        jacoco_trigger_command=jacoco_trigger_command,
        static_du_trigger_command=static_du_trigger_command,
        jacoco_trigger_success=jacoco_trigger_result.exit_code == 0,
        static_du_trigger_success=static_du_trigger_result.exit_code == 0,
        jacoco_json=repo_path / "jacoco.json",
        static_du_json=repo_path / "static_du.json",
        def_use_json=repo_path / "def_use.json" if (repo_path / "def_use.json").exists() else None,
        unified_trigger=False,
    )
    return status, jacoco_console, jacoco_trigger_console, static_du_trigger_console


def load_json(path: Path | None) -> dict[str, Any]:
    if path is None or not path.exists():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


def metric_rows_by_name(payload: dict[str, Any]) -> dict[str, dict[str, Any]]:
    rows = payload.get("metrics", [])
    mapping: dict[str, dict[str, Any]] = {}
    if isinstance(rows, list):
        for row in rows:
            if isinstance(row, dict):
                name = str(row.get("l5_metric", ""))
                if name:
                    mapping[name] = row
    return mapping


def nested_get(payload: dict[str, Any], *keys: str) -> Any:
    current: Any = payload
    for key in keys:
        if not isinstance(current, dict) or key not in current:
            return None
        current = current[key]
    return current


def def_use_summary(jacoco_json: dict[str, Any]) -> dict[str, Any]:
    summary = nested_get(jacoco_json, "supplemental_raw_data", "static_du_summary")
    return summary if isinstance(summary, dict) else {}


def summary_value(summary: dict[str, Any], canonical_key: str) -> Any:
    for alias in DEF_USE_SUMMARY_KEYS.get(canonical_key, [canonical_key]):
        if alias in summary:
            return summary[alias]
    return None


def platform_proxy_disclosure(metric: str) -> str:
    return PLATFORM_PROXY_MAP.get(metric, "")


def assess_evidence_quality(evidence: str, metric: str, directly_emitted: str) -> str:
    if not evidence or not str(evidence).strip():
        return "Missing"
    weak_metrics = {
        "CI/CD Quality Gate Enforcement",
        "Cross-Function Use Detection",
        "Reporting Validation",
        "Coverage Reporting Validation",
        "Edge Case Handling",
    }
    if metric in weak_metrics:
        return "Weak"
    if directly_emitted == "Yes":
        return "Strong"
    if platform_proxy_disclosure(metric):
        return "Strong"
    return "Weak"


def classify_coverage_tier(directly_emitted: str, derived: str, supported: str) -> str:
    if supported not in {"Supported", "Baseline Not Available", "Partially Supported"}:
        return "Not_Supported"
    if directly_emitted == "Yes":
        return "Native"
    if derived == "Yes" or supported in {"Baseline Not Available", "Partially Supported"}:
        return "Platform_Derived"
    return "Not_Supported"


def resolve_supported_with_disclosure(
    *,
    native_supported: bool,
    platform_present: bool,
    evidence: str,
    metric: str,
) -> tuple[str, str, str, str]:
    proxy = platform_proxy_disclosure(metric)
    if native_supported:
        return "Supported", "Yes", "No", ""
    if platform_present and proxy:
        quality = assess_evidence_quality(evidence, metric, "No")
        if quality == "Missing":
            return "Partially Supported", "No", "Yes", proxy
        return "Supported", "No", "Yes", proxy
    if platform_present:
        return "Partially Supported", "No", "Yes", proxy or "platform score without documented proxy"
    return "Not Supported", "No", "No", ""


def compute_branch_alignment(jacoco_xml: Path | None, jacoco_json: dict[str, Any]) -> dict[str, Any]:
    xml_branch = xml_total_counter_percent(jacoco_xml, "BRANCH") if jacoco_xml else None
    platform_summary = jacoco_json.get("summary", {}) if isinstance(jacoco_json.get("summary"), dict) else {}
    platform_branch = platform_summary.get("branch_percent")
    discrepancy = False
    delta: float | None = None
    if xml_branch is not None and platform_branch is not None:
        delta = round(float(platform_branch) - float(xml_branch), 4)
        discrepancy = abs(delta) > 0.01
    return {
        "xml_branch_percent": xml_branch,
        "platform_branch_percent": platform_branch,
        "branch_percent_delta": delta,
        "branch_percent_discrepancy": "Yes" if discrepancy else "No",
        "discrepancy_detail": (
            f"Platform jacoco.json branch_percent={platform_branch} differs from native jacoco.xml BRANCH={xml_branch}%"
            if discrepancy
            else "Platform summary matches native XML branch percent within tolerance."
        ),
    }


def validation_lookup(*dfs: pd.DataFrame) -> dict[str, dict[str, str]]:
    lookup: dict[str, dict[str, str]] = {}
    for frame in dfs:
        if frame is None or frame.empty or "Metric" not in frame.columns:
            continue
        for row in frame.to_dict("records"):
            metric = str(row.get("Metric", ""))
            if metric:
                lookup[metric] = row
    return lookup


def build_repo_routing_csv(output_csv: Path) -> pd.DataFrame:
    frame = pd.DataFrame(REPO_ROUTING_ROWS)
    frame.to_csv(output_csv, index=False)
    return frame


def build_trigger_manifest(output_dir: Path) -> pd.DataFrame:
    frame = pd.DataFrame(TAXONOMY_TRIGGER_MANIFEST)
    frame.to_csv(output_dir / "taxonomy_trigger_manifest.csv", index=False)
    (output_dir / "taxonomy_trigger_manifest.json").write_text(
        json.dumps(TAXONOMY_TRIGGER_MANIFEST, indent=2),
        encoding="utf-8",
    )
    return frame


def build_metric_coverage_action_plan(taxonomy_truth_df: pd.DataFrame, output_csv: Path) -> pd.DataFrame:
    rows: list[dict[str, str]] = []
    for _, truth in taxonomy_truth_df.iterrows():
        metric = str(truth["Metric"])
        alt = METRIC_ALTERNATIVE_MAP.get(metric, {})
        native = str(truth.get("Coverage_Tier", "")) == "Native"
        platform_only = str(truth.get("Coverage_Tier", "")) == "Platform_Derived"
        rows.append(
            {
                "Testing_Type": truth.get("Testing_Type", ""),
                "Classification": truth.get("Classification", ""),
                "Metric": metric,
                "Goal": truth.get("Goal", ""),
                "Current_Repo_Triggers_Tool": "Yes",
                "Current_Repo_Native_Coverage": "Yes" if native else "No",
                "Current_Repo_Platform_Only": "Yes" if platform_only else "No",
                "Current_Status": "OK native" if native else ("Platform proxy only — not real measurement" if platform_only else "Not covered"),
                "Alternative_Tool": alt.get("alternative_tool", ""),
                "Alternative_Repo": alt.get("alternative_repo", ""),
                "Alternative_Pipeline": alt.get("alternative_pipeline", ""),
                "Required_Data_To_Add": alt.get("required_data", ""),
                "Native_Output_Fields": alt.get("native_output_fields", ""),
                "Run_Command": PIPELINE_RUN_COMMANDS.get(alt.get("alternative_pipeline", ""), ""),
            }
        )
    frame = pd.DataFrame(rows)
    frame.to_csv(output_csv, index=False)
    return frame


def build_taxonomy_truth_table(
    control_flow_df: pd.DataFrame,
    coverage_delta_df: pd.DataFrame,
    data_flow_df: pd.DataFrame,
    jacoco_xml: Path | None,
    jacoco_json: dict[str, Any],
    output_csv: Path,
) -> pd.DataFrame:
    lookup = validation_lookup(control_flow_df, coverage_delta_df, data_flow_df)
    branch_alignment = compute_branch_alignment(jacoco_xml, jacoco_json)
    rows: list[dict[str, str]] = []

    for testing_type, classification, metric, goal in TAXONOMY_TRUTH_ROWS:
        validation_row = lookup.get(metric, {})
        supported = str(validation_row.get("Supported", "Not Supported"))
        directly = str(validation_row.get("Directly Emitted", "No"))
        derived = str(validation_row.get("Derived", "No"))
        evidence = str(validation_row.get("Evidence", ""))
        proxy = str(validation_row.get("Proxy_Disclosure", platform_proxy_disclosure(metric)))
        coverage_tier = classify_coverage_tier(directly, derived, supported)
        evidence_quality = assess_evidence_quality(evidence, metric, directly)

        recommended = next(
            (
                row
                for row in REPO_ROUTING_ROWS
                if metric in row.get("Taxonomy_Section", "")
                or (
                    testing_type.startswith("Control Flow")
                    and row["Metric_Family"] == "Control Flow / Path Coverage"
                )
                or (
                    testing_type.startswith("Test Regression")
                    and row["Metric_Family"] == "Coverage Regression / Delta"
                )
                or (
                    testing_type.startswith("Data Flow")
                    and row["Metric_Family"] == "Data Flow / Definition-Use"
                )
            ),
            REPO_ROUTING_ROWS[-1],
        )

        root_cause = ""
        if coverage_tier == "Native":
            root_cause = "Emitted directly by official tool output."
        elif coverage_tier == "Platform_Derived":
            root_cause = "Training-repo platform wrapper derives score; not native JaCoCo/Static DU schema."
        else:
            root_cause = "No native or disclosed platform evidence found."

        rows.append(
            {
                "Testing_Type": testing_type,
                "Classification": classification,
                "Metric": metric,
                "Goal": goal,
                "Supported": supported,
                "Coverage_Tier": coverage_tier,
                "Evidence_Quality": evidence_quality,
                "Directly_Emitted": directly,
                "Derived": derived,
                "Proxy_Disclosure": proxy,
                "Artifact": str(validation_row.get("Artifact", validation_row.get("Tool", ""))),
                "Evidence": evidence[:500],
                "Recommended_Pipeline": recommended["Recommended_Pipeline"],
                "Recommended_Repo": recommended["Recommended_Repo"],
                "Root_Cause": root_cause,
            }
        )

    frame = pd.DataFrame(rows)
    frame.attrs["branch_alignment"] = branch_alignment
    frame.to_csv(output_csv, index=False)
    return frame


def build_extended_jacoco_metrics_csv(xml_path: Path, output_csv: Path) -> pd.DataFrame:
    counters = parse_counter_map(xml_path)
    rows = counters_to_metrics_rows(counters)
    for counter_type in sorted(JACOCO_COUNTER_TYPES):
        values = counters.get(counter_type, {"missed": 0, "covered": 0})
        rows.append(
            {
                "metric_name": f"{counter_type} Counter",
                "covered": values.get("covered", 0),
                "missed": values.get("missed", 0),
                "coverage_percent": coverage_percent(values.get("covered", 0), values.get("missed", 0)),
            }
        )
    frame = pd.DataFrame(rows)
    frame.to_csv(output_csv, index=False)
    return frame


def build_static_du_metrics_csv(
    static_du_json: dict[str, Any],
    output_csv: Path,
    jacoco_json: dict[str, Any] | None = None,
) -> pd.DataFrame:
    rows: list[dict[str, str]] = []
    summary = static_du_json.get("summary", {})
    supplemental = nested_get(static_du_json, "supplemental_raw_data", "static_du_summary")
    if not isinstance(summary, dict):
        summary = {}
    if not isinstance(supplemental, dict):
        supplemental = {}

    def add_rows(source: dict[str, Any], artifact: str, emission: str = "Directly Emitted") -> None:
        for key, value in source.items():
            rows.append(
                {
                    "metric_name": str(key),
                    "metric_value": str(value),
                    "artifact": artifact,
                    "emission_type": emission,
                }
            )

    add_rows(summary, "static_du.json:summary")
    add_rows(supplemental, "static_du.json:supplemental_raw_data.static_du_summary")
    jacoco_supplemental = nested_get(jacoco_json or {}, "supplemental_raw_data", "static_du_summary")
    if isinstance(jacoco_supplemental, dict):
        add_rows(jacoco_supplemental, "jacoco.json:supplemental_raw_data.static_du_summary", "Derived")
    for row in static_du_json.get("metrics", []) if isinstance(static_du_json.get("metrics"), list) else []:
        if not isinstance(row, dict):
            continue
        rows.append(
            {
                "metric_name": str(row.get("l5_metric", "")),
                "metric_value": str(row.get("score", "")),
                "artifact": "static_du.json:metrics",
                "emission_type": "Directly Emitted",
            }
        )
    frame = pd.DataFrame(rows)
    frame.to_csv(output_csv, index=False)
    return frame


def validate_control_flow_metrics(
    jacoco_xml: Path | None,
    jacoco_json: dict[str, Any],
    keyword_df: pd.DataFrame,
) -> pd.DataFrame:
    xml_validation = validate_path_metrics(
        keyword_df,
        {
            "jacoco.xml": jacoco_xml,
            "jacoco.csv": jacoco_xml.parent / "jacoco.csv" if jacoco_xml else None,
        },
        jacoco_xml,
    )
    xml_map = {row["Metric"]: row for row in xml_validation.to_dict("records")}
    platform_map = metric_rows_by_name(jacoco_json)
    rows: list[dict[str, str]] = []

    for metric in CONTROL_FLOW_METRICS:
        xml_row = xml_map.get(metric, {})
        platform_row = platform_map.get(metric, {})
        xml_supported = xml_row.get("Supported") == "Supported"
        platform_present = bool(platform_row)
        if xml_supported:
            supported = "Supported"
            directly = "Yes"
            derived = "No"
            artifact = xml_row.get("Artifact", "jacoco.xml")
            evidence = xml_row.get("Comments", "")
            proxy = ""
        elif platform_present:
            raw_params = platform_row.get("raw_parameters", {})
            evidence = (
                f"l5_metric={metric}; score={platform_row.get('score', '')}; "
                f"jacoco_native={platform_row.get('jacoco_native', '')}; "
                f"raw_parameters={raw_params}"
            )[:500]
            supported, directly, derived, proxy = resolve_supported_with_disclosure(
                native_supported=False,
                platform_present=True,
                evidence=evidence,
                metric=metric,
            )
            artifact = "jacoco.json"
        elif xml_row.get("Supported") == "Not Supported":
            supported = "Not Supported"
            directly = "No"
            derived = "No"
            artifact = "jacoco.xml"
            evidence = xml_row.get("Comments", "")
            proxy = ""
        else:
            supported = "Not Supported"
            directly = "No"
            derived = "No"
            artifact = ""
            evidence = "No explicit control-flow metric evidence found in JaCoCo XML or platform JSON."
            proxy = ""

        rows.append(
            {
                "Metric": metric,
                "Classification": "Path Coverage",
                "Tool": "JaCoCo",
                "Supported": supported,
                "Coverage_Tier": classify_coverage_tier(directly, derived, supported),
                "Evidence_Quality": assess_evidence_quality(evidence, metric, directly),
                "Directly Emitted": directly,
                "Derived": derived,
                "Proxy_Disclosure": proxy,
                "Artifact": artifact,
                "Evidence": evidence,
                "Comments": "Native JaCoCo XML has no PATH counter; platform jacoco.json scores are branch/line proxies.",
            }
        )
    return pd.DataFrame(rows)


def xml_total_counter_percent(xml_path: Path, counter_type: str) -> float | None:
    if not xml_path.exists():
        return None
    counters = parse_counter_map(xml_path)
    values = counters.get(counter_type)
    if not values:
        return None
    return coverage_percent(values.get("covered", 0), values.get("missed", 0))


def validate_coverage_delta_metrics(
    current_xml: Path | None,
    baseline_xml: Path | None,
    jacoco_json: dict[str, Any],
) -> pd.DataFrame:
    rows: list[dict[str, str]] = []
    platform_summary = jacoco_json.get("summary", {}) if isinstance(jacoco_json.get("summary"), dict) else {}
    platform_map = metric_rows_by_name(jacoco_json)

    baseline_available = baseline_xml is not None and baseline_xml.exists()
    computed_delta: dict[str, float | None] = {}
    if baseline_available and current_xml and current_xml.exists():
        for counter in ("INSTRUCTION", "LINE", "BRANCH"):
            current = xml_total_counter_percent(current_xml, counter)
            baseline = xml_total_counter_percent(baseline_xml, counter)
            if current is not None and baseline is not None:
                computed_delta[counter] = round(current - baseline, 4)

    metric_evidence = {
        "Coverage Delta": ["coverage_delta_percent"],
        "Regression Testing Monitoring": ["modules_tested", "modules_with_churn"],
        "Test Suite Effectiveness Tracking": ["line_percent", "branch_percent", "instruction_percent"],
        "CI/CD Quality Gate Enforcement": ["metric_coverage_complete", "metrics_covered"],
        "Change Impact Analysis": ["modules_with_churn", "coverage_delta_percent"],
        "Quality Improvement Measurement": ["coverage_delta_percent", "line_percent"],
    }

    for metric in COVERAGE_REGRESSION_METRICS:
        platform_row = platform_map.get(metric, {})
        summary_hits = []
        for key in metric_evidence.get(metric, []):
            if key in platform_summary:
                summary_hits.append(f"{key}={platform_summary.get(key)}")
            elif key in jacoco_json:
                summary_hits.append(f"{key}={jacoco_json.get(key)}")
        if platform_row and not summary_hits:
            summary_hits.append(f"score={platform_row.get('score', '')}")
        has_xml_delta = bool(computed_delta) and metric == "Coverage Delta"
        platform_present = bool(platform_row or summary_hits)

        evidence_parts = list(summary_hits)
        if has_xml_delta:
            evidence_parts.append(f"computed_xml_delta={computed_delta}")
        evidence = "; ".join(evidence_parts)[:500]

        if has_xml_delta and not platform_present:
            supported = "Supported"
            directly = "Yes"
            derived = "No"
            proxy = "computed_xml_delta from jacoco.xml baseline comparison"
            artifact = "jacoco.xml"
            comments = "Coverage delta computed from native XML baseline comparison."
        elif platform_present:
            supported, directly, derived, proxy = resolve_supported_with_disclosure(
                native_supported=has_xml_delta,
                platform_present=True,
                evidence=evidence,
                metric=metric,
            )
            if has_xml_delta:
                directly = "Yes"
                derived = "Yes" if platform_present else "No"
                supported = "Supported"
                proxy = f"{proxy}; computed_xml_delta from jacoco.xml" if proxy else "computed_xml_delta from jacoco.xml"
            artifact = "jacoco.json+jacoco.xml" if has_xml_delta else "jacoco.json"
            comments = "Platform JSON summary/metrics checked; optional XML baseline comparison performed when baseline exists."
        elif not baseline_available:
            supported = "Baseline Not Available"
            directly = "No"
            derived = "No"
            proxy = ""
            artifact = ""
            evidence = "Baseline Not Available"
            comments = "No baseline XML supplied or found; coverage delta not computed from XML comparison."
        else:
            supported = "Not Supported"
            directly = "No"
            derived = "No"
            proxy = ""
            artifact = ""
            evidence = ""
            comments = "No explicit coverage regression evidence found."

        rows.append(
            {
                "Metric": metric,
                "Supported": supported,
                "Coverage_Tier": classify_coverage_tier(directly, derived, supported),
                "Evidence_Quality": assess_evidence_quality(evidence, metric, directly),
                "Directly Emitted": directly,
                "Derived": derived,
                "Proxy_Disclosure": proxy,
                "Artifact": artifact,
                "Evidence": evidence,
                "Comments": comments,
            }
        )
    return pd.DataFrame(rows)


def metric_tool_evidence(
    metric: str,
    jacoco_json: dict[str, Any],
    static_du_json: dict[str, Any],
    def_use: dict[str, Any],
) -> tuple[str, str, str, str, str, str]:
    jacoco_row = metric_rows_by_name(jacoco_json).get(metric, {})
    static_du_row = metric_rows_by_name(static_du_json).get(metric, {})
    tools: list[str] = []
    evidence_parts: list[str] = []

    if jacoco_row:
        tools.append("JaCoCo")
        evidence_parts.append(
            f"jacoco.json score={jacoco_row.get('score', '')} raw_parameters={jacoco_row.get('raw_parameters', {})}"[:250]
        )
    if static_du_row:
        tools.append("Static DU")
        evidence_parts.append(
            f"static_du.json score={static_du_row.get('score', '')} raw_parameters={static_du_row.get('raw_parameters', {})}"[:250]
        )

    def_use_map = {
        "Variable Definition Detection": ["definitions_total", "definitions_covered", "all_defs_percent"],
        "Definition-Use Mapping": ["du_pairs_total", "du_pairs_covered", "data_path_correlation_percent"],
        "Coverage Measurement": ["du_path_percent", "all_defs_percent", "all_uses_percent"],
        "Uncovered Definition Detection": ["uncovered_definitions"],
        "Edge Case Handling": ["ghost_uses", "partial_uses"],
        "Reporting Validation": ["metrics_total", "metrics_covered"],
        "Computational Use Detection (C-Use)": ["c_use_total", "c_use_covered", "c_use_percent"],
        "Predicate Use Detection (P-Use)": ["p_use_total", "p_use_covered", "p_use_percent"],
        "Definition-Use Pair Identification": ["du_pairs_total", "du_paths"],
        "All-Uses Coverage Verification": ["all_uses_percent", "uses_total", "uses_covered"],
        "Partial Uses Coverage Detection": ["partial_uses"],
        "Multiple Definitions Handling": ["multiple_definition_sites"],
        "Cross-Function Use Detection": ["cross_function_uses"],
        "Unreachable Use Detection": ["ghost_uses"],
        "Coverage Reporting Validation": ["metrics_covered", "metric_coverage_complete"],
        "Variable Use Detection": ["uses_total", "uses_covered", "all_uses_percent"],
    }
    for key in def_use_map.get(metric, []):
        value = summary_value(def_use, key) if key in DEF_USE_SUMMARY_KEYS else def_use.get(key)
        if value is None and isinstance(jacoco_json.get("summary"), dict):
            value = jacoco_json["summary"].get(key)
        if value is not None:
            if "JaCoCo" not in tools:
                tools.append("JaCoCo")
            evidence_parts.append(f"{key}={value}")

    tool = " + ".join(tools) if tools else "None"
    evidence = " | ".join(evidence_parts)[:500]
    analysis_mode = nested_get(static_du_json, "supplemental_raw_data", "static_du_meta", "analysis_mode")
    has_def_use_fields = any(
        part
        for part in evidence_parts
        if any(
            token in part
            for token in (
                "definitions_",
                "uses_",
                "c_use_",
                "p_use_",
                "du_pairs",
                "ghost_uses",
                "partial_uses",
                "multiple_definition",
            )
        )
    )
    native_static_du = (
        bool(static_du_row)
        and analysis_mode not in (None, "static_code_duplication")
        and has_def_use_fields
    )
    platform_present = bool(jacoco_row or (def_use and has_def_use_fields))
    if tools:
        supported, directly, derived, proxy = resolve_supported_with_disclosure(
            native_supported=native_static_du,
            platform_present=platform_present,
            evidence=evidence,
            metric=metric,
        )
        if static_du_row and analysis_mode == "static_code_duplication":
            tool = "JaCoCo platform def-use (standalone Static DU emits duplication only)"
    else:
        supported = "Not Supported"
        directly = "No"
        derived = "No"
        proxy = ""
    return supported, directly, derived, tool, evidence, proxy


def validate_data_flow_metrics(jacoco_json: dict[str, Any], static_du_json: dict[str, Any]) -> pd.DataFrame:
    def_use = def_use_summary(jacoco_json)
    rows: list[dict[str, str]] = []
    for testing_type, classification, metric in DATA_FLOW_METRICS:
        supported, directly, derived, tool, evidence, proxy = metric_tool_evidence(
            metric, jacoco_json, static_du_json, def_use
        )
        analysis_mode = nested_get(static_du_json, "supplemental_raw_data", "static_du_meta", "analysis_mode")
        has_native_static_du = analysis_mode not in (None, "static_code_duplication") and "Static DU" in tool
        primary_tool = "Static DU" if has_native_static_du else "JaCoCo"
        supporting_tool = "JaCoCo" if primary_tool == "Static DU" else ("Static DU" if "static_du" in tool.lower() else "")
        if analysis_mode == "static_code_duplication":
            primary_tool = "JaCoCo"
            supporting_tool = "Static DU (duplication only)"
        evidence_file = "jacoco.json"
        if "static_du.json" in evidence and "jacoco.json" not in evidence:
            evidence_file = "static_du.json"
        elif "static_du.json" in evidence and "jacoco.json" in evidence:
            evidence_file = "jacoco.json+static_du.json"
        rows.append(
            {
                "Testing Type": testing_type,
                "Classification": classification,
                "Metric": metric,
                "Primary Tool": primary_tool,
                "Supporting Tool": supporting_tool,
                "Tool": tool,
                "Supported": supported,
                "Coverage_Tier": classify_coverage_tier(directly, derived, supported),
                "Evidence_Quality": assess_evidence_quality(evidence, metric, directly),
                "Directly Emitted": directly,
                "Derived": derived,
                "Proxy_Disclosure": proxy,
                "Evidence File": evidence_file,
                "Evidence Value": evidence,
                "Evidence": evidence,
                "Comments": "Definition-use evidence comes from jacoco-platform StaticDuAnalyzer; standalone static_du.json emits duplication metrics in this repository.",
            }
        )
    return pd.DataFrame(rows)


def count_tests(java_files: list[Path]) -> int:
    return sum(1 for path in java_files if "/test/" in str(path).replace("\\", "/") or "\\test\\" in str(path))


def count_methods(java_files: list[Path]) -> int:
    total = 0
    for path in java_files:
        if "/test/" in str(path).replace("\\", "/"):
            continue
        try:
            text = path.read_text(encoding="utf-8")
        except OSError:
            continue
        total += sum(1 for line in text.splitlines() if " void " in line and "(" in line and ";" not in line.split("(")[0])
    return total


def build_combined_repository_summary(
    java_files: list[Path],
    jacoco_xml: Path | None,
    jacoco_json: dict[str, Any],
    static_du_json: dict[str, Any],
) -> pd.DataFrame:
    counters = parse_counter_map(jacoco_xml) if jacoco_xml and jacoco_xml.exists() else {}
    def_use = def_use_summary(jacoco_json)
    packages = {extract_package_name(path) for path in java_files if extract_package_name(path)}
    main_files = [path for path in java_files if "/test/" not in str(path).replace("\\", "/")]
    row = {
        "Java Files": len(java_files),
        "Packages": len(packages),
        "Classes": len({path.stem for path in main_files}),
        "Methods": count_methods(java_files),
        "Tests": count_tests(java_files),
        "Definitions": summary_value(def_use, "definitions_total") or "",
        "Uses": summary_value(def_use, "uses_total") or "",
        "DU Pairs": summary_value(def_use, "du_pairs_total") or "",
        "Instruction Counters": counters.get("INSTRUCTION", {}).get("covered", ""),
        "Branch Counters": counters.get("BRANCH", {}).get("covered", ""),
        "Complexity Counters": counters.get("COMPLEXITY", {}).get("covered", ""),
        "Static DU Total Lines": nested_get(static_du_json, "summary", "total_lines") or "",
        "JaCoCo Metrics Covered": jacoco_json.get("metrics_covered", ""),
        "Static DU Metrics Covered": static_du_json.get("metrics_covered", ""),
    }
    return pd.DataFrame([row])


def copy_platform_json_artifacts(repo_path: Path, output_dir: Path) -> dict[str, bool]:
    copied: dict[str, bool] = {}
    for name in PLATFORM_JSON_ARTIFACTS:
        source = repo_path / name
        if source.exists():
            copy_artifact(source, output_dir / name)
            copied[name] = True
        else:
            copied[name] = False
    if copied.get("static_du.json") and not (output_dir / "static_du_output.json").exists():
        copy_artifact(repo_path / "static_du.json", output_dir / "static_du_output.json")
    return copied


def build_cross_validation(
    repo_path: Path,
    output_dir: Path,
    jacoco_xml: Path | None,
    jacoco_json: dict[str, Any],
    static_du_json: dict[str, Any],
) -> pd.DataFrame:
    rows: list[dict[str, str]] = []

    if jacoco_xml and jacoco_xml.exists():
        xml_counters = parse_counter_map(jacoco_xml)
        for counter_type in JACOCO_COUNTER_TYPES:
            values = xml_counters.get(counter_type, {})
            expected = coverage_percent(values.get("covered", 0), values.get("missed", 0))
            rows.append(
                {
                    "Metric": f"JaCoCo XML {counter_type} percent",
                    "Expected Value": str(expected),
                    "Observed Value": str(expected),
                    "Match": "Yes",
                    "Artifact": "jacoco.xml",
                }
            )

    jacoco_metrics_path = output_dir / "jacoco_metrics.csv"
    if jacoco_metrics_path.exists():
        jacoco_df = pd.read_csv(jacoco_metrics_path)
        for _, row in jacoco_df.iterrows():
            rows.append(
                {
                    "Metric": str(row.get("metric_name", "")),
                    "Expected Value": "present in jacoco.xml",
                    "Observed Value": str(row.get("coverage_percent", row.get("covered", ""))),
                    "Match": "Yes" if row.get("metric_name") else "No",
                    "Artifact": "jacoco_metrics.csv",
                }
            )

    static_du_metrics_file = output_dir / "static_du_metrics.json"
    if not static_du_metrics_file.exists():
        static_du_metrics_file = repo_path / "static_du_metrics.json"
    if static_du_metrics_file.exists():
        payload = load_json(static_du_metrics_file)
        if isinstance(payload, dict):
            for key, value in payload.items():
                rows.append(
                    {
                        "Metric": f"static_du_metrics.{key}",
                        "Expected Value": str(value),
                        "Observed Value": str(value),
                        "Match": "Yes",
                        "Artifact": "static_du_metrics.json",
                    }
                )

    metrics_json = load_json(output_dir / "metrics.json") if (output_dir / "metrics.json").exists() else load_json(repo_path / "metrics.json")
    dashboard_json = load_json(output_dir / "dashboard_metrics.json") if (output_dir / "dashboard_metrics.json").exists() else {}
    platform_json = load_json(output_dir / "platform_metrics.json") if (output_dir / "platform_metrics.json").exists() else {}

    if isinstance(metrics_json, dict) and isinstance(dashboard_json, dict):
        for key in ("metrics_total", "metrics_covered", "metric_coverage_complete"):
            expected = metrics_json.get(key, dashboard_json.get(key))
            observed = dashboard_json.get(key)
            if expected is not None or observed is not None:
                rows.append(
                    {
                        "Metric": f"dashboard.{key}",
                        "Expected Value": str(expected),
                        "Observed Value": str(observed),
                        "Match": "Yes" if str(expected) == str(observed) else "No",
                        "Artifact": "metrics.json+dashboard_metrics.json",
                    }
                )

    if isinstance(metrics_json, dict) and isinstance(platform_json, dict):
        for key in set(metrics_json.keys()) & set(platform_json.keys()):
            rows.append(
                {
                    "Metric": f"platform.{key}",
                    "Expected Value": str(metrics_json.get(key)),
                    "Observed Value": str(platform_json.get(key)),
                    "Match": "Yes" if metrics_json.get(key) == platform_json.get(key) else "No",
                    "Artifact": "metrics.json+platform_metrics.json",
                }
            )

    for label, payload, artifact in (
        ("jacoco.json metrics_covered", jacoco_json, "jacoco.json"),
        ("static_du.json metrics_covered", static_du_json, "static_du.json"),
    ):
        covered = payload.get("metrics_covered")
        if covered is not None:
            rows.append(
                {
                    "Metric": label,
                    "Expected Value": str(covered),
                    "Observed Value": str(covered),
                    "Match": "Yes",
                    "Artifact": artifact,
                }
            )

    frame = pd.DataFrame(rows)
    frame.to_csv(output_dir / "cross_validation.csv", index=False)
    return frame


def build_dashboard_summary(
    repo_path: Path,
    build_tool: str,
    status: CombinedRunStatus,
    control_flow_df: pd.DataFrame,
    coverage_delta_df: pd.DataFrame,
    data_flow_df: pd.DataFrame,
    java_files: list[Path],
) -> pd.DataFrame:
    def supported_count(frame: pd.DataFrame) -> int:
        return int(frame["Supported"].isin(["Supported", "Partially Supported"]).sum())

    def unsupported_count(frame: pd.DataFrame) -> int:
        return int((frame["Supported"] == "Not Supported").sum())

    total_supported = supported_count(control_flow_df) + supported_count(coverage_delta_df) + supported_count(data_flow_df)
    total_unsupported = unsupported_count(control_flow_df) + unsupported_count(coverage_delta_df) + unsupported_count(data_flow_df)

    row = {
        "Repository": repo_path.name,
        "Build Tool": build_tool,
        "Tests Executed": count_tests(java_files),
        "JaCoCo Status": "OK" if status.build_status.report_generated else "FAILED",
        "Static DU Status": "OK" if status.static_du_trigger_success else "FAILED",
        "Control Flow Metrics Supported": supported_count(control_flow_df),
        "Regression Metrics Supported": supported_count(coverage_delta_df),
        "Data Flow Metrics Supported": supported_count(data_flow_df),
        "Total Metrics Supported": total_supported,
        "Unsupported Metrics": total_unsupported,
        "Unified DefUse Trigger": "Yes" if status.unified_trigger else "No",
    }
    frame = pd.DataFrame([row])
    return frame


def export_validation_bundle(output_dir: Path, parsed: dict[str, Any]) -> Path:
    bundle = {
        "control_flow_validation": parsed["control_flow_df"].to_dict("records"),
        "coverage_delta_validation": parsed["coverage_delta_df"].to_dict("records"),
        "data_flow_validation": parsed["data_flow_df"].to_dict("records"),
        "cross_validation": parsed.get("cross_validation_df", pd.DataFrame()).to_dict("records"),
        "taxonomy_truth_table": parsed.get("taxonomy_truth_df", pd.DataFrame()).to_dict("records"),
        "dashboard_summary": parsed.get("dashboard_summary_df", pd.DataFrame()).to_dict("records"),
        "repository_summary": parsed.get("repository_summary_df", pd.DataFrame()).to_dict("records"),
    }
    path = output_dir / "validation_results.json"
    path.write_text(json.dumps(bundle, indent=2), encoding="utf-8")
    return path


def build_dashboard_metrics(
    repo_path: Path,
    build_tool: str,
    java_files: list[Path],
    status: CombinedRunStatus,
    control_flow_df: pd.DataFrame,
    coverage_delta_df: pd.DataFrame,
    data_flow_df: pd.DataFrame,
    jacoco_xml: Path | None = None,
    jacoco_json: dict[str, Any] | None = None,
    taxonomy_truth_df: pd.DataFrame | None = None,
) -> pd.DataFrame:
    repo_summary = build_combined_repository_summary(
        java_files,
        status.build_status.jacoco_xml,
        jacoco_json or load_json(status.jacoco_json),
        load_json(status.static_du_json),
    ).iloc[0]
    branch_alignment = compute_branch_alignment(jacoco_xml, jacoco_json or {})

    def supported_count(frame: pd.DataFrame) -> int:
        return int(frame["Supported"].isin(["Supported", "Partially Supported"]).sum())

    rows: list[dict[str, Any]] = [
        {"Metric": "Repository", "Value": repo_path.name},
        {"Metric": "Build Tool", "Value": build_tool},
        {"Metric": "Java Files", "Value": repo_summary["Java Files"]},
        {"Metric": "Packages", "Value": repo_summary["Packages"]},
        {"Metric": "Classes", "Value": repo_summary["Classes"]},
        {"Metric": "Methods", "Value": repo_summary["Methods"]},
        {"Metric": "Tests Executed", "Value": repo_summary["Tests"]},
        {"Metric": "JaCoCo Status", "Value": "OK" if status.build_status.report_generated else "FAILED"},
        {"Metric": "Static DU Status", "Value": "OK" if status.static_du_trigger_success else "FAILED"},
        {"Metric": "Control Flow Metrics Supported", "Value": supported_count(control_flow_df)},
        {"Metric": "Coverage Regression Metrics Supported", "Value": supported_count(coverage_delta_df)},
        {"Metric": "Data Flow Metrics Supported", "Value": supported_count(data_flow_df)},
        {
            "Metric": "Unsupported Metrics",
            "Value": int((control_flow_df["Supported"] == "Not Supported").sum())
            + int((coverage_delta_df["Supported"] == "Not Supported").sum())
            + int((data_flow_df["Supported"] == "Not Supported").sum()),
        },
        {"Metric": "JaCoCo XML Branch Percent", "Value": branch_alignment["xml_branch_percent"]},
        {"Metric": "Platform jacoco.json Branch Percent", "Value": branch_alignment["platform_branch_percent"]},
        {"Metric": "Branch Percent Discrepancy", "Value": branch_alignment["branch_percent_discrepancy"]},
        {"Metric": "Branch Percent Delta (Platform - XML)", "Value": branch_alignment["branch_percent_delta"]},
        {"Metric": "Branch Discrepancy Detail", "Value": branch_alignment["discrepancy_detail"]},
    ]

    if taxonomy_truth_df is not None and not taxonomy_truth_df.empty:
        rows.extend(
            [
                {
                    "Metric": "Taxonomy Rows Native Tier",
                    "Value": int((taxonomy_truth_df["Coverage_Tier"] == "Native").sum()),
                },
                {
                    "Metric": "Taxonomy Rows Platform Derived Tier",
                    "Value": int((taxonomy_truth_df["Coverage_Tier"] == "Platform_Derived").sum()),
                },
                {
                    "Metric": "Taxonomy Rows Not Supported Tier",
                    "Value": int((taxonomy_truth_df["Coverage_Tier"] == "Not_Supported").sum()),
                },
                {
                    "Metric": "Taxonomy Rows Weak Evidence",
                    "Value": int((taxonomy_truth_df["Evidence_Quality"] == "Weak").sum()),
                },
                {
                    "Metric": "Taxonomy Rows Missing Evidence",
                    "Value": int((taxonomy_truth_df["Evidence_Quality"] == "Missing").sum()),
                },
            ]
        )

    return pd.DataFrame(rows)


def collect_all_outputs(
    status: CombinedRunStatus,
    repo_path: Path,
    java_files: list[Path],
    build_tool: str,
    output_dir: Path,
    baseline_xml: Path | None,
    jacoco_console: str,
    jacoco_trigger_console: str,
    static_du_trigger_console: str,
) -> dict[str, Any]:
    ensure_output_dir(output_dir)
    copied = copy_raw_jacoco_artifacts(status.build_status, output_dir)
    static_du_copied = {
        "static_du_output.json": copy_artifact(status.static_du_json, output_dir / "static_du_output.json"),
    }
    training = repo_path / "artifacts" / "training"
    for name in ("static_du_summary.json", "du_path_correlation.json", "static_du_meta.json"):
        candidate = training / name
        if candidate.exists():
            copy_artifact(candidate, output_dir / name)
    for pattern in ("static_du*.xml", "static_du*.csv", "*static_du*.csv", "*static_du*.xml"):
        for candidate in repo_path.rglob(pattern.split("/")[-1]):
            if not candidate.is_file():
                continue
            suffix = candidate.suffix.lower()
            if suffix == ".xml":
                static_du_copied["static_du_output.xml"] = copy_artifact(candidate, output_dir / "static_du_output.xml")
            elif suffix == ".csv":
                static_du_copied["static_du_output.csv"] = copy_artifact(candidate, output_dir / "static_du_output.csv")

    jacoco_console_path = output_dir / "jacoco_console_output.txt"
    jacoco_console_path.write_text(
        "\n\n".join(
            [
                "===== MAVEN BUILD / JACOCO =====",
                jacoco_console,
                "===== JACOCO PLATFORM TRIGGER =====",
                jacoco_trigger_console,
            ]
        ),
        encoding="utf-8",
    )
    static_du_console_path = output_dir / "static_du_console_output.txt"
    static_du_console_path.write_text(static_du_trigger_console, encoding="utf-8")

    jacoco_json = load_json(status.jacoco_json)
    static_du_json = load_json(status.static_du_json)
    jacoco_xml = output_dir / "jacoco.xml" if copied.get("jacoco.xml") else status.build_status.jacoco_xml

    platform_json_copied = copy_platform_json_artifacts(repo_path, output_dir)

    jacoco_metrics_df = build_extended_jacoco_metrics_csv(jacoco_xml, output_dir / "jacoco_metrics.csv")
    static_du_metrics_df = build_static_du_metrics_csv(
        static_du_json, output_dir / "static_du_metrics.csv", jacoco_json=jacoco_json
    )

    artifacts = {
        "jacoco_console_output.txt": jacoco_console_path,
        "jacoco.xml": output_dir / "jacoco.xml",
        "jacoco.csv": output_dir / "jacoco.csv",
        "index.html": output_dir / "index.html",
    }
    keyword_df = search_path_keywords(artifacts)
    control_flow_df = validate_control_flow_metrics(jacoco_xml, jacoco_json, keyword_df)
    control_flow_df.to_csv(output_dir / "control_flow_validation.csv", index=False)

    baseline_path = baseline_xml if baseline_xml and baseline_xml.exists() else repo_path / "artifacts" / "training" / "baseline_jacoco.xml"
    coverage_delta_df = validate_coverage_delta_metrics(jacoco_xml, baseline_path, jacoco_json)
    coverage_delta_df.to_csv(output_dir / "coverage_delta_validation.csv", index=False)

    data_flow_df = validate_data_flow_metrics(jacoco_json, static_du_json)
    data_flow_df.to_csv(output_dir / "data_flow_validation.csv", index=False)

    repo_routing_df = build_repo_routing_csv(output_dir / "repo_routing.csv")
    trigger_manifest_df = build_trigger_manifest(output_dir)
    taxonomy_truth_df = build_taxonomy_truth_table(
        control_flow_df,
        coverage_delta_df,
        data_flow_df,
        jacoco_xml,
        jacoco_json,
        output_dir / "taxonomy_truth_table.csv",
    )
    action_plan_df = build_metric_coverage_action_plan(
        taxonomy_truth_df,
        output_dir / "metric_coverage_action_plan.csv",
    )

    repository_summary_df = build_combined_repository_summary(java_files, jacoco_xml, jacoco_json, static_du_json)
    repository_summary_df.to_csv(output_dir / "repository_summary.csv", index=False)

    dashboard_df = build_dashboard_metrics(
        repo_path,
        build_tool,
        java_files,
        status,
        control_flow_df,
        coverage_delta_df,
        data_flow_df,
        jacoco_xml=jacoco_xml,
        jacoco_json=jacoco_json,
        taxonomy_truth_df=taxonomy_truth_df,
    )
    dashboard_df.to_csv(output_dir / "dashboard_metrics.csv", index=False)

    dashboard_summary_df = build_dashboard_summary(
        repo_path, build_tool, status, control_flow_df, coverage_delta_df, data_flow_df, java_files
    )
    dashboard_summary_df.to_csv(output_dir / "dashboard_summary.csv", index=False)

    cross_validation_df = build_cross_validation(repo_path, output_dir, jacoco_xml, jacoco_json, static_du_json)

    parsed_bundle = {
        "control_flow_df": control_flow_df,
        "coverage_delta_df": coverage_delta_df,
        "data_flow_df": data_flow_df,
        "cross_validation_df": cross_validation_df,
        "taxonomy_truth_df": taxonomy_truth_df,
        "dashboard_summary_df": dashboard_summary_df,
        "repository_summary_df": repository_summary_df,
    }
    validation_json_path = export_validation_bundle(output_dir, parsed_bundle)

    return {
        "copied": copied,
        "static_du_copied": static_du_copied,
        "platform_json_copied": platform_json_copied,
        "jacoco_metrics_df": jacoco_metrics_df,
        "static_du_metrics_df": static_du_metrics_df,
        "control_flow_df": control_flow_df,
        "coverage_delta_df": coverage_delta_df,
        "data_flow_df": data_flow_df,
        "repo_routing_df": repo_routing_df,
        "trigger_manifest_df": trigger_manifest_df,
        "taxonomy_truth_df": taxonomy_truth_df,
        "action_plan_df": action_plan_df,
        "repository_summary_df": repository_summary_df,
        "dashboard_df": dashboard_df,
        "dashboard_summary_df": dashboard_summary_df,
        "cross_validation_df": cross_validation_df,
        "validation_results_json": validation_json_path,
    }



## Section 4 — Repository Setup


In [ ]:
OUTPUT_PATH = Path(OUTPUT_DIR).resolve()
WORKSPACE_PATH = Path(WORKSPACE_DIR).resolve()
ERROR_LOG_PATH = OUTPUT_PATH / 'error_log.txt'

ensure_output_dir(OUTPUT_PATH)
logger = NotebookLogger(ERROR_LOG_PATH)
JAVA_ENV = configure_java_runtime(logger)
JAVA_VERSION = java_version_text(JAVA_ENV)

REPO_PATH = resolve_repository_path(
    use_git_repo=USE_GIT_REPO,
    repo_url=REPO_URL,
    local_repo=LOCAL_REPO,
    workspace_dir=WORKSPACE_PATH,
    if_clone_exists=IF_CLONE_EXISTS,
    logger=logger,
    clone_depth=CLONE_DEPTH,
)

BUILD_TOOL = detect_build_tool(REPO_PATH)
JAVA_FILES = discover_java_files(REPO_PATH)
if not JAVA_FILES:
    raise FileNotFoundError('No Java source files found.')

save_java_inventory(JAVA_FILES, OUTPUT_PATH / 'java_files_inventory.csv')
print(f'Repository: {REPO_PATH.name}')
print(f'Build Tool: {BUILD_TOOL}')
print(f'Java Files: {len(JAVA_FILES)}')


## Section 5 — Detect Build Tool


In [ ]:
print('Detected Build Tool:', BUILD_TOOL)


## Section 6 — Build Project, Execute JaCoCo + Static DU (Unified DefUseTrigger)


In [ ]:
PIPELINE_STARTED = time.perf_counter()
COMBINED_STATUS, JACOCO_BUILD_CONSOLE, JACOCO_TRIGGER_CONSOLE, STATIC_DU_CONSOLE = execute_platform_triggers(
    REPO_PATH, JAVA_ENV, logger, skip_verify=SKIP_VERIFY
)
print('Unified trigger:', COMBINED_STATUS.unified_trigger)
print('Build success:', COMBINED_STATUS.build_status.build_success)
print('JaCoCo report:', COMBINED_STATUS.build_status.report_generated)
print('JaCoCo trigger:', COMBINED_STATUS.jacoco_trigger_success)
print('Static DU trigger:', COMBINED_STATUS.static_du_trigger_success)


## Section 7 — Preserve Raw Outputs


In [ ]:
BASELINE_PATH = Path(BASELINE_JACOCO_XML).resolve() if BASELINE_JACOCO_XML else None
TOTAL_EXECUTION_TIME = round(time.perf_counter() - PIPELINE_STARTED, 5)
PARSED = collect_all_outputs(
    COMBINED_STATUS, REPO_PATH, JAVA_FILES, BUILD_TOOL, OUTPUT_PATH,
    BASELINE_PATH, JACOCO_BUILD_CONSOLE, JACOCO_TRIGGER_CONSOLE, STATIC_DU_CONSOLE,
)
print('JaCoCo artifacts:', PARSED['copied'])
print('Platform JSON:', PARSED['platform_json_copied'])
print('Validation JSON:', PARSED['validation_results_json'])


## Section 8 — Parse JaCoCo


In [ ]:
display(PARSED['jacoco_metrics_df'])


## Section 9 — Parse Static DU


In [ ]:
display(PARSED['static_du_metrics_df'].head(25))


## Section 10 — Validate Control Flow Testing


In [ ]:
CONTROL_FLOW_DF = PARSED['control_flow_df']
display(CONTROL_FLOW_DF)


## Section 11 — Validate Test Regression / Coverage Analysis


In [ ]:
COVERAGE_DELTA_DF = PARSED['coverage_delta_df']
display(COVERAGE_DELTA_DF)


## Section 12 — Validate Data Flow Testing


In [ ]:
DATA_FLOW_DF = PARSED['data_flow_df']
display(DATA_FLOW_DF)


## Section 13 — Cross Validation


In [ ]:
CROSS_VALIDATION_DF = PARSED['cross_validation_df']
display(CROSS_VALIDATION_DF.head(20))


## Section 14 — Taxonomy Truth Table


In [ ]:
display(PARSED['taxonomy_truth_df'].head(15))
print(PARSED['taxonomy_truth_df']['Coverage_Tier'].value_counts())


## Section 15 — Repository Summary


In [ ]:
display(PARSED['repository_summary_df'])


## Section 16 — Dashboard


In [ ]:
display(PARSED['dashboard_summary_df'])
display(PARSED['dashboard_df'])
print(f'Total execution time (s): {TOTAL_EXECUTION_TIME}')


## Section 17 — JSON Exports


In [ ]:
import json
for name in ['jacoco.json', 'static_du.json', 'def_use.json', 'metrics.json', 'platform_metrics.json', 'dashboard_metrics.json', 'validation_results.json']:
    path = OUTPUT_PATH / name
    print(f'{name}:', 'OK' if path.exists() else 'MISSING')
with open(OUTPUT_PATH / 'validation_results.json', encoding='utf-8') as fh:
    preview = json.load(fh)
print('validation_results.json keys:', list(preview.keys()))


## Section 18 — Error Handling


In [ ]:
logger.write_errors()
if ERROR_LOG_PATH.exists() and ERROR_LOG_PATH.stat().st_size:
    display(pd.read_csv(ERROR_LOG_PATH))
else:
    print('No errors logged.')
